# AETHER Qwen Brain — Kaggle Deployment

Runs **Qwen3.8-27B** and serves it to AETHER behind three endpoints
(`/chat`, `/generate`, `/director`) over an ngrok tunnel.

**Before you run anything:** set the accelerator to **GPU T4 x2**
(Settings → Accelerator), and add `NGROK_AUTHTOKEN` and `DIRECTOR_API_KEY`
under Add-ons → Secrets.

### Why llama.cpp and not vLLM

Kaggle's GPUs are Turing (T4, compute capability 7.5), and this model's
compressed-tensors 4-bit build — `cyankiwi/Qwen3.8-27B-AWQ-INT4`, which is not
AWQ despite the name — decodes through Marlin kernels that require sm_80. It
cannot load on a T4 under vLLM at any setting.

llama.cpp is a different stack: its CUDA kernels run on Turing, and GGUF is an
unrelated quantization format. So the model runs here as a GGUF quantization
from `unsloth/Qwen3.8-27B-GGUF`.

Two things follow from that choice:

* llama.cpp splits a model across GPUs **by layer, not tensor-parallel**. The
  second T4 buys capacity, not speed — the cards take turns.
* There is one model in one process and it is not thread-safe, so requests are
  serialized behind a lock rather than batched.

## Stage 1 — Hardware

In [ ]:
import subprocess, sys, torch

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

GPU_COUNT = torch.cuda.device_count()
if GPU_COUNT == 0:
    raise RuntimeError('No GPU. Settings -> Accelerator -> GPU T4 x2.')

for i in range(GPU_COUNT):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory/1024**3:.1f} GB  sm_{p.major}{p.minor}')

TOTAL_VRAM = sum(torch.cuda.get_device_properties(i).total_memory
                 for i in range(GPU_COUNT)) / 1024**3

print(f'\nCUDA {torch.version.cuda} | torch {torch.__version__}')
print(f'GPUs {GPU_COUNT} | total VRAM {TOTAL_VRAM:.1f} GB')
if GPU_COUNT < 2:
    print('\n[WARN] One GPU only. Q5_K_M (19.8 GB) will spill to CPU and crawl.')
    print('       This is what a Colab free session looks like; Kaggle gives 2x T4.')
    print('       Drop to Q3_K_M in Stage 3 if you are staying here.')
print('\nStage 1 PASSED')

## Stage 2 — Dependencies (~5 min; the CUDA wheel is 1.9 GB)

In [ ]:
import subprocess, sys, threading, time

# Run pip and keep talking while it works. -q with no heartbeat is why this
# cell looks hung: pulling a 1.9 GB wheel takes minutes and prints nothing,
# which is indistinguishable from a dead kernel. Anything here that can run
# longer than a few seconds says so.
def pip(*args, label=''):
    print(f'  [pip] {label or args[0]} ...', flush=True)
    started = time.time()
    done = threading.Event()

    def beat():
        while not done.wait(20):
            print(f'        still working ({int(time.time()-started)}s)', flush=True)

    threading.Thread(target=beat, daemon=True).start()
    proc = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q', *args],
        capture_output=True, text=True,
    )
    done.set()

    if proc.returncode != 0:
        print(proc.stdout[-3000:])
        print(proc.stderr[-3000:])
        raise RuntimeError(f'pip failed for {label or args[0]} (exit {proc.returncode})')
    print(f'        done in {int(time.time()-started)}s', flush=True)

# The prebuilt wheel links against libcudart.so.12, so the matching CUDA
# runtime packages have to be present or importing llama_cpp dies with
# "libcudart.so.12: cannot open shared object file" — even though the GPU is
# fine and nvidia-smi works.
pip('nvidia-cuda-runtime-cu12', 'nvidia-cublas-cu12', label='CUDA runtime')

# --no-deps matters more than it looks. A plain --force-reinstall drags every
# dependency with it, which on this image means swapping numpy out from under
# a kernel that already imported torch in Stage 1 — and that kills the kernel
# later, at a random cell, with no error pointing back here.
#
# --index-url, not --extra-index-url. With --extra-index-url, PyPI stays the
# primary index and pip picks the best candidate across both — and PyPI ships
# llama-cpp-python as an sdist ONLY. So when no wheel on the CUDA index matches
# this image, pip quietly falls back to the source tarball and starts compiling
# llama.cpp, which takes far longer than the download it replaced and looks
# exactly like a hung cell. --no-deps means nothing else is being resolved, so
# restricting the index to the CUDA one is safe.
#
# --only-binary=:all: is the belt to that braces: if there is no usable wheel,
# fail immediately and say so, rather than starting an hour-long build.
CUDA_WHEEL_INDEX = 'https://abetlen.github.io/llama-cpp-python/whl/cu124'

# The wheels changed platform tag partway through the 0.3 series: up to 0.3.19
# they are cp3XX-linux_x86_64, which installs on any Linux, and from 0.3.30 they
# are py3-none-manylinux_2_35, which needs glibc 2.35 or newer. So the newest
# release is not automatically the installable one on this image.
import platform
print('  glibc:', '.'.join(platform.libc_ver()[1].split('.')[:2]) or 'unknown')

try:
    pip('--force-reinstall', '--no-deps', '--only-binary=:all:', 'llama-cpp-python',
        '--index-url', CUDA_WHEEL_INDEX,
        label='llama-cpp-python CUDA wheel (1.9 GB — the slow one, 3-8 min)')
except RuntimeError as exc:
    # No wheel matched. Almost always the manylinux_2_35 tag against an older
    # glibc; fall back to the last release tagged plain linux_x86_64, which
    # installs regardless.
    print(f'\n  no compatible wheel for the newest release ({exc})')
    print('  falling back to 0.3.19, the last release tagged linux_x86_64\n', flush=True)
    pip('--force-reinstall', '--no-deps', '--only-binary=:all:', 'llama-cpp-python==0.3.19',
        '--index-url', CUDA_WHEEL_INDEX,
        label='llama-cpp-python 0.3.19 CUDA wheel')

# Its actual dependencies, installed normally so anything already satisfied is
# left alone.
pip('diskcache', 'jinja2', 'typing-extensions',
    'fastapi', 'uvicorn[standard]', 'pyngrok', 'huggingface_hub', 'pydantic',
    label='supporting packages')

import importlib.metadata as md
for pkg in ('llama-cpp-python', 'fastapi', 'pydantic', 'huggingface_hub', 'numpy'):
    try:
        print(f'  {pkg:22} {md.version(pkg)}')
    except Exception:
        print(f'  {pkg:22} (not installed)')
print('\nStage 2 PASSED')

## Stage 3 — Configuration

In [ ]:
import os

# Sizes are the download, and roughly the VRAM the weights occupy.
#
#   quant      size     2x T4 (30 GB)              1x T4 (15 GB)
#   Q3_K_M     13.8 GB  lots of headroom           just fits
#   Q4_K_M     17.1 GB  comfortable                spills to CPU
#   Q5_K_M     19.8 GB  fits  <- default           spills badly
#   Q8_0       29.0 GB  no room for the KV cache   no
MODEL_REPO = 'unsloth/Qwen3.8-27B-GGUF'
QUANT      = 'Q5_K_M'
# MODEL_FILE is resolved in Stage 6 from the repo listing, because the
# published filenames change when the model is re-quantized.
MODEL_DIR  = '/tmp/qwen38'

# This model is hybrid: only about a quarter of its 64 layers use full
# attention, and the rest are linear-attention layers with a fixed-size state.
# So the KV cache costs roughly 32 KB per token rather than 128 KB, and 16k of
# context is about half a gigabyte. Raising this is cheaper than it looks.
N_CTX    = 16384
API_PORT = 8001

# Reading the two secrets is a network call to Kaggle's API, and it prints
# nothing while it waits. On a slow response this cell looks identical to a hung
# kernel, which is how it came to be interrupted mid-request — the traceback
# pointed at ssl.read, which reads like a crash and is really just a stop.
#
# So: say what is being fetched, and name each one as it lands.
NGROK_AUTHTOKEN  = ''
DIRECTOR_API_KEY = ''
try:
    from kaggle_secrets import UserSecretsClient

    print('Reading secrets from Add-ons -> Secrets (a network call; a few seconds)...',
          flush=True)
    _s = UserSecretsClient()
    for _label in ('NGROK_AUTHTOKEN', 'DIRECTOR_API_KEY'):
        try:
            _value = _s.get_secret(_label)
            if _label == 'NGROK_AUTHTOKEN':
                NGROK_AUTHTOKEN = _value
            else:
                DIRECTOR_API_KEY = _value
            print(f'  {_label:<18} found', flush=True)
        except Exception as _exc:  # one missing secret must not lose the other
            print(f'  {_label:<18} not set ({type(_exc).__name__})', flush=True)
except Exception as exc:
    print(f'Kaggle secrets unavailable ({exc}); falling back to environment.', flush=True)

# Whatever the secrets store did not supply, the environment might.
NGROK_AUTHTOKEN  = NGROK_AUTHTOKEN  or os.environ.get('NGROK_AUTHTOKEN', '')
DIRECTOR_API_KEY = DIRECTOR_API_KEY or os.environ.get('DIRECTOR_API_KEY', 'test-key-change-me')

if not NGROK_AUTHTOKEN:
    print('\n  [WARN] No NGROK_AUTHTOKEN. Stage 10 will serve on localhost only,')
    print('         so AETHER on your machine will not be able to reach this.')

os.environ['DIRECTOR_MODEL']   = f'{MODEL_REPO}:{QUANT}'
os.environ['DIRECTOR_API_KEY'] = DIRECTOR_API_KEY
os.environ['DIRECTOR_N_CTX']   = str(N_CTX)

print('Model   :', MODEL_REPO, QUANT)
print('Context :', f'{N_CTX:,} tokens')
print('API key :', (DIRECTOR_API_KEY[:4] + '****') if DIRECTOR_API_KEY else 'NOT SET')
print('Ngrok   :', 'configured' if NGROK_AUTHTOKEN else 'NOT SET (localhost only)')

## Stage 4 — Write the director package

In [ ]:
import os
from pathlib import Path
os.makedirs('director', exist_ok=True)
print('[stage 4] cwd:', os.getcwd())

In [ ]:
# director/__init__.py — generated from the package by build_notebook.py
SRC = '# Initialize director module\n'
Path('director/__init__.py').write_text(SRC, encoding='utf-8')
print('  wrote director/__init__.py', len(SRC), 'bytes')

In [ ]:
# director/schema.py — generated from the package by build_notebook.py
SRC = '"""The storyboard contract.\n\nThis mirrors, field for field, what AETHER\'s `parseVideoPlan()` in\npublic/prompts.js actually reads. That parser is unforgiving in two ways worth\nstating up front, because getting either wrong makes the director look broken\nin a way that produces no error message:\n\n  * a scene whose `index` is not a finite number is DROPPED from the result,\n    so an omitted index silently deletes the beat; and\n  * `visualType` is snapped onto AETHER\'s vocabulary, with anything\n    unrecognised falling back to \'stock_video\' — an invented type is not an\n    error, it is a wrong answer that looks like a right one.\n\nThese models are compiled into a GBNF grammar by llama.cpp, so the grammar\nitself is what keeps the model inside the vocabulary — it cannot emit a token\nthat leaves the shape. Widening a Literal here widens what the model may say,\nso the lists below must stay in step with VISUAL_TYPES / SHOT_TYPES /\nCAMERA_MOVES / HOST_OVERLAYS in public/prompts.js.\n\nNote that llama.cpp builds the grammar and never shows the schema to the\nmodel, so the Field descriptions here do not steer generation. Field guidance\nbelongs in DIRECTOR_SYSTEM_PROMPT; the descriptions are for whoever reads\nthis file next.\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom functools import lru_cache\nfrom typing import Annotated, Any, List, Literal, Optional, Tuple, Union\n\nfrom pydantic import BaseModel, Field, model_validator\n\n# Which payload each visual type cannot render without. A grammar cannot say\n# "if visualType is stock_video then stockRequirements is required" — JSON\n# Schema has no way to express a dependency between a value and another\n# field\'s presence — so the model is free to pick a type and then leave its\n# payload null. It does exactly that when given the chance, and the beat\n# renders as a blank card. This table is where the rule actually lives.\nNEEDS_STOCK = frozenset({"stock_video", "stock_photo", "stock_text"})\nNEEDS_TEXT = frozenset({"stock_text", "editorial_text"})\nNEEDS_GRAPHIC = frozenset({"stickman", "whiteboard", "chart", "map", "timeline", "diagram"})\n\n# Kept in the same order as public/prompts.js so the two can be diffed by eye.\nVisualType = Literal[\n    "stock_video", "stock_photo", "stock_text", "editorial_text",\n    "t2v", "broll", "presenter",\n    "stickman", "whiteboard", "chart", "map", "timeline", "diagram",\n]\n\nShotType = Literal[\n    "Extreme Wide", "Wide", "Medium", "Close Up",\n    "Extreme Close Up", "Over Shoulder", "POV",\n]\n\nCameraMove = Literal[\n    "Static", "Slow Push In", "Dolly In", "Dolly Out", "Pan Left", "Pan Right",\n    "Crane Up", "Crane Down", "Handheld", "Drone",\n]\n\nHostOverlay = Literal["none", "circle", "rect", "corner", "full"]\nTextStyle = Literal["stat", "quote", "title", "emphasis", "callout"]\nTransition = Literal["cut", "dissolve"]\n\n\nSourceStrategy = Literal["auto", "modern_stock", "archival", "mixed"]\nMediaSource = Literal["pexels", "pixabay", "archive_org"]\n\n\nclass TimePeriod(BaseModel):\n    """When the beat is set, when the script actually says so.\n\n    Only fill this in from something the narration establishes. A guessed date\n    narrows the archive search onto the wrong decade, which is worse than not\n    narrowing it at all.\n    """\n\n    from_year: Optional[int] = Field(default=None, ge=1800, le=2100,\n                                     description="Earliest year this beat depicts.")\n    to_year: Optional[int] = Field(default=None, ge=1800, le=2100,\n                                   description="Latest year this beat depicts.")\n    label: str = Field(default="", description="How the script says it: \'1960s\', \'the Cold War\'.")\n\n\nclass Excerpt(BaseModel):\n    """Which slice of a long archive item this beat wants.\n\n    Archive items are whole films, so a beat needs a window rather than the\n    whole reel. `selectionIntent` says what should be on screen; AETHER finds\n    the window and flags it for checking, because choosing the right moment\n    inside an eleven-minute film means watching it.\n    """\n\n    required: bool = Field(default=False, description="True when the source is a long-form film.")\n    targetDuration: float = Field(default=6.0, ge=1, le=60,\n                                  description="Seconds of it to use.")\n    selectionIntent: str = Field(\n        default="",\n        description="What should be visible in the excerpt: \'workers operating wartime machinery\'.",\n    )\n\n\nclass _StockCommon(BaseModel):\n    """What every stock beat needs, whichever library it is bound for."""\n\n    concept: str = Field(\n        description="One sentence naming what this beat is actually about."\n    )\n    queries: List[str] = Field(\n        description=(\n            "3-5 standalone stock-search phrases, written as a stock "\n            "photographer would search: subject + action + setting. Describe "\n            "what the camera sees, never what the narration says. Good: "\n            "\'scientist looking into microscope\'. Bad: \'the consequences of "\n            "inflation\'."\n        ),\n        min_length=1,\n        max_length=6,\n    )\n    fallbackQueries: List[str] = Field(\n        default_factory=list,\n        description="2-3 broader queries to try if the primary queries find nothing.",\n        max_length=4,\n    )\n    subjectCategory: Optional[Literal["HUMAN", "NATURE", "URBAN", "ABSTRACT", "OBJECT"]] = Field(\n        default=None, description="Coarse subject bucket, used to break ties between clips."\n    )\n    minimumDuration: float = Field(\n        default=0,\n        ge=0,\n        description="Seconds the clip must run at minimum. 0 means half the scene duration.",\n    )\n    preferredSources: List[MediaSource] = Field(\n        min_length=1,\n        max_length=3,\n        description=(\n            "REQUIRED, best first. Name at least one — an empty list is not "\n            "\'let AETHER choose\', it is a beat with no source decision at all. "\n            "List each library once."\n        ),\n    )\n    sourceReason: str = Field(\n        default="",\n        description=(\n            "One line on why this source suits the beat, for the producer to "\n            "read. \'Authentic 1969 launch footage carries this better than a "\n            "modern re-creation.\' Not your reasoning — the production rationale."\n        ),\n    )\n\n\nclass ArchivalRequirements(_StockCommon):\n    """An archival beat, and everything AETHER needs to actually use one.\n\n    A film archive is not a stock library with older clips in it. Asking it for\n    footage means naming the artefact, choosing a window inside a whole film,\n    and saying why that film earns its place. Those are four separate decisions\n    and the Director is the only thing that can make them — so on this branch\n    they are structurally required, not encouraged.\n\n    Required here rather than validated afterwards because a validation failure\n    re-asks the model for the entire plan, and one beat costs about 150 seconds\n    against the live 27B. The grammar can simply make the omission unspellable.\n\n    `sourceStrategy` is first so the grammar\'s alternation resolves on its very\n    first field: once the model has written "archival" it is committed to this\n    branch and cannot reach the end of the object without the rest.\n    """\n\n    sourceStrategy: Literal["archival"] = Field(\n        description=(\n            "Authentic period film says more here than any modern restaging: a "\n            "war, a launch, a protest, a vanished way of working."\n        ),\n    )\n    archiveQueries: List[str] = Field(\n        min_length=1,\n        max_length=3,\n        description=(\n            "REQUIRED. Searches phrased for a film archive rather than a stock "\n            "library. An archive is catalogued by what a film IS, not by what "\n            "it shows: \'1940s wartime factory newsreel\' finds something, \'happy "\n            "worker in factory\' does not."\n        ),\n    )\n    excerpt: Excerpt = Field(\n        description=(\n            "REQUIRED. An archive item is a whole film, so the beat needs a "\n            "window into it and a statement of what should be visible there."\n        ),\n    )\n    editorialPurpose: str = Field(\n        min_length=1,\n        description=(\n            "REQUIRED. Why this footage serves the story — \'historical context "\n            "for the wartime production figures\'. A production note, not a "\n            "legal claim: never assert that a use is fair, permitted or safe. "\n            "Whether uncleared material may be published is decided by a "\n            "person, not here."\n        ),\n    )\n    timePeriod: Optional[TimePeriod] = Field(\n        default=None,\n        description=(\n            "Fill this in whenever the narration establishes a date, and leave "\n            "it out when it does not. A guessed decade points the search at the "\n            "wrong footage, which is worse than not narrowing it at all."\n        ),\n    )\n\n    @model_validator(mode="after")\n    def _archive_is_among_the_sources(self) -> "ArchivalRequirements":\n        # Repaired rather than rejected. The model has already said this beat is\n        # archival; a preferredSources list that then omits archive_org is a\n        # slip, not a different decision, and rejecting it would cost a full\n        # re-plan. "contains" is not expressible in JSON Schema, so this is the\n        # one archival rule the grammar cannot carry.\n        if "archive_org" not in self.preferredSources:\n            self.preferredSources = ["archive_org"] + list(self.preferredSources)[:2]\n        return self\n\n\nclass ModernRequirements(_StockCommon):\n    """A present-day beat. The archival fields are absent because they are meaningless here.\n\n    Nothing in a modern stock library needs an excerpt window or an archive\n    query, and requiring them would teach the model to invent both.\n    """\n\n    sourceStrategy: Literal["modern_stock", "auto"] = Field(\n        description=(\n            "\'modern_stock\' for present-day life; \'auto\' only when either kind "\n            "would genuinely serve — not as a way to avoid choosing."\n        ),\n    )\n\n\n# Discriminated on sourceStrategy, so the grammar itself decides which fields a\n# beat must carry. This is what makes "required for archival, irrelevant for\n# modern" expressible at all: a single flat model can only say "optional".\nStockRequirements = Annotated[\n    Union[ArchivalRequirements, ModernRequirements],\n    Field(discriminator="sourceStrategy"),\n]\n\n\nclass TextOverlay(BaseModel):\n    """Editorial type burned over the beat. Required for stock_text and editorial_text."""\n\n    text: str = Field(\n        description="The words on screen. Under 12 words — this is a caption, not the narration."\n    )\n    emphasis: str = Field(\n        default="",\n        description="The 1-3 words within `text` to set larger or brighter.",\n    )\n    style: TextStyle = Field(default="emphasis", description="Which type treatment to use.")\n\n\nclass Graphic(BaseModel):\n    """Content for a canvas-drawn beat: stickman, whiteboard, chart, map, timeline, diagram."""\n\n    title: str = Field(default="", description="Heading for the card.")\n    subtitle: str = Field(default="", description="Optional second line.")\n    items: List[str] = Field(\n        default_factory=list,\n        max_length=6,\n        description=(\n            "The actual content to typeset, and the format depends on the type: "\n            "\'Label: Number\' pairs for chart, \'Date: Event\' for timeline, place "\n            "names for map, steps for whiteboard, labelled parts for diagram, and "\n            "\'action:expression\' pairs such as \'explain:confident\' for stickman."\n        ),\n    )\n\n\nclass Scene(BaseModel):\n    """One beat. AETHER calls these scenes and they map 1:1 onto storyboard rows."""\n\n    # -- timing ---------------------------------------------------------------\n    # Present only when the narration has been transcribed. These are seconds\n    # on the FINISHED video\'s clock, and they must come from the supplied\n    # transcript rather than being estimated from sentence length: application\n    # code validates them against the audio duration and rejects a shot that\n    # starts before zero, ends after the narration, or overlaps its neighbour.\n    #\n    # Not to be confused with an archival excerpt\'s sourceIn/sourceOut, which\n    # are positions inside the source film. Two clocks, never interchangeable.\n    timelineStart: Optional[float] = Field(\n        default=None, ge=0,\n        description="When this beat appears, in seconds of the finished video.",\n    )\n    timelineEnd: Optional[float] = Field(\n        default=None, ge=0,\n        description="When it leaves. Must be greater than timelineStart.",\n    )\n\n    index: int = Field(\n        ge=0,\n        description=(\n            "Zero-based position of this beat, matching the input cue list. "\n            "AETHER discards any scene without a numeric index, so never omit it."\n        ),\n    )\n    visualType: VisualType = Field(description="Which renderer this beat is sent to.")\n\n    stockRequirements: Optional[StockRequirements] = Field(\n        default=None, description="Required for stock_video, stock_photo and stock_text."\n    )\n    textOverlay: Optional[TextOverlay] = Field(\n        default=None, description="Required for stock_text and editorial_text."\n    )\n    graphic: Optional[Graphic] = Field(\n        default=None,\n        description="Required for stickman, whiteboard, chart, map, timeline and diagram.",\n    )\n\n    hostOverlay: HostOverlay = Field(\n        default="none",\n        description=(\n            "Where the channel host sits over the visual. \'full\' only with "\n            "presenter; \'none\' for footage beats and anything emotional."\n        ),\n    )\n    shotType: ShotType = Field(default="Medium", description="Shot scale.")\n    cameraMovement: CameraMove = Field(default="Static", description="Camera move.")\n    motion: str = Field(\n        default="",\n        description="What physically moves in the shot. A footage beat with no motion is an expensive still.",\n    )\n    emotion: str = Field(default="", description="One or two words for the intended mood.")\n    transition: Transition = Field(\n        default="cut", description="How this beat joins the NEXT one."\n    )\n    note: str = Field(\n        default="",\n        description="Continuity warnings, or why this visual choice was made. Keep it short.",\n    )\n\n    @model_validator(mode="after")\n    def _payload_matches_type(self) -> "Scene":\n        problems = scene_violations(self.model_dump())\n        if problems:\n            raise ValueError("; ".join(problems))\n        return self\n\n\nclass VideoPlan(BaseModel):\n    """The whole director answer. Top-level shape read by parseVideoPlan()."""\n\n    strategy: str = Field(\n        default="",\n        description="A sentence or two on the visual approach taken across the video.",\n    )\n    warnings: List[str] = Field(\n        default_factory=list,\n        description="Continuity problems: contradictions in place, time of day or weather.",\n    )\n    scenes: List[Scene] = Field(min_length=1, description="Every beat, in order.")\n\n\n# Retained so `from .schema import Storyboard` keeps working in older notebooks.\nStoryboard = VideoPlan\n\n\ndef scene_violations(scene: dict) -> List[str]:\n    """What is wrong with one scene, in words a model can act on.\n\n    Takes a plain dict so it can run on raw model output before Pydantic gets\n    a chance to reject it — the repair path needs to see the damage to fix it.\n    """\n    problems: List[str] = []\n    visual = scene.get("visualType")\n\n    index = scene.get("index")\n    if not isinstance(index, int) or isinstance(index, bool):\n        # AETHER silently drops a scene with no numeric index, so this is the\n        # difference between a beat and no beat at all.\n        problems.append("index is missing or not a whole number")\n\n    requirements = scene.get("stockRequirements") or {}\n    queries = requirements.get("queries") or []\n    if visual in NEEDS_STOCK and not queries:\n        problems.append(f"{visual} needs stockRequirements.queries — there is nothing to search for")\n\n    if visual in NEEDS_STOCK:\n        sources = requirements.get("preferredSources") or []\n        if not sources:\n            # Left empty, nothing routes this beat and every library gets the\n            # same generic search. Measured on the live model: with this field\n            # optional, it was empty on every beat including a wartime one.\n            problems.append(\n                f"{visual} needs stockRequirements.preferredSources — name at least one library"\n            )\n        if not requirements.get("sourceStrategy"):\n            problems.append(f"{visual} needs stockRequirements.sourceStrategy")\n\n    # Deliberately NOT a violation: archive_org preferred with no archiveQueries.\n    #\n    # It is a real quality problem — an archive is catalogued by what a film IS,\n    # so stock phrasing searches it badly — but scene_violations means "this beat\n    # cannot be rendered", and this one renders perfectly well: StockMedia falls\n    # back to the ordinary queries. Listing it here made every such beat fail\n    # validation and triggered a full re-generation of the whole plan, which is\n    # a large cost for a beat that was never broken. See archive_query_advice().\n\n    if visual in NEEDS_TEXT and not ((scene.get("textOverlay") or {}).get("text") or "").strip():\n        problems.append(f"{visual} needs textOverlay.text — it would render as a blank card")\n\n    if visual in NEEDS_GRAPHIC and not ((scene.get("graphic") or {}).get("items")):\n        problems.append(f"{visual} needs graphic.items — it would render as a blank card")\n\n    return problems\n\n\ndef archive_query_advice(plan: dict) -> List[Tuple[Any, str]]:\n    """Quality notes that are worth showing but not worth regenerating for.\n\n    Separated from plan_violations because the two have different costs: a\n    violation re-asks the model for the entire plan, while these are things a\n    producer can see and fix in one click.\n    """\n    notes: List[Tuple[Any, str]] = []\n    for position, scene in enumerate(plan.get("scenes") or []):\n        if not isinstance(scene, dict):\n            continue\n        requirements = scene.get("stockRequirements") or {}\n        sources = requirements.get("preferredSources") or []\n        if "archive_org" in sources and not (requirements.get("archiveQueries") or []):\n            notes.append((\n                scene.get("index", position),\n                "archive_org is preferred but archiveQueries is empty — the archive "\n                "will be searched with stock phrasing, which finds far less",\n            ))\n    return notes\n\n\ndef plan_violations(plan: dict) -> List[Tuple[Any, str]]:\n    """Every problem across a plan, as (index, message) pairs."""\n    found: List[Tuple[Any, str]] = []\n    for position, scene in enumerate(plan.get("scenes") or []):\n        if not isinstance(scene, dict):\n            found.append((position, "scene is not an object"))\n            continue\n        for problem in scene_violations(scene):\n            found.append((scene.get("index", position), problem))\n    return found\n\n\ndef _tighten(node: object) -> object:\n    """Forbid unspecified keys everywhere in the generated JSON schema.\n\n    Guided decoding follows the grammar it is handed. Left open, an object\n    permits arbitrary extra keys, and the model will occasionally invent one\n    instead of filling the field we asked for — which then arrives as a beat\n    with no queries. Closing the objects removes the option.\n    """\n    if isinstance(node, dict):\n        if node.get("type") == "object" and "additionalProperties" not in node:\n            node["additionalProperties"] = False\n        for value in node.values():\n            _tighten(value)\n    elif isinstance(node, list):\n        for value in node:\n            _tighten(value)\n    return node\n\n\ndef _inline_refs(node: object, defs: dict, depth: int = 0) -> object:\n    """Replace every $ref with the definition it points at.\n\n    llama.cpp compiles the schema into a GBNF grammar, and its converter is\n    far less tolerant of `$ref`/`$defs` indirection than a validator is. The\n    models here are a plain tree with no recursion, so inlining is safe and\n    costs only a slightly larger schema.\n    """\n    if depth > 32:  # a cycle would otherwise expand forever\n        raise ValueError("schema nests deeper than expected — is a model recursive?")\n\n    if isinstance(node, dict):\n        ref = node.get("$ref")\n        if isinstance(ref, str) and ref.startswith("#/$defs/"):\n            target = dict(defs[ref.split("/")[-1]])\n            # Keep any siblings of the $ref (a description, usually).\n            merged = {k: v for k, v in node.items() if k != "$ref"}\n            target.update(merged)\n            return _inline_refs(target, defs, depth + 1)\n        return {k: _inline_refs(v, defs, depth + 1) for k, v in node.items() if k != "$defs"}\n\n    if isinstance(node, list):\n        return [_inline_refs(v, defs, depth + 1) for v in node]\n\n    return node\n\n\n# Annotations that mean something to a validator but nothing to a grammar.\n# `minimum` in particular is compiled inconsistently across llama.cpp versions,\n# and a numeric bound is not worth risking the whole grammar over — the\n# Pydantic model still enforces it on the way back out.\n_GRAMMAR_NOISE = ("description", "title", "default", "minimum", "maximum",\n                  "exclusiveMinimum", "exclusiveMaximum",\n                  # A discriminated union carries a mapping of literal values to\n                  # "#/$defs/..." strings. Those are VALUES, so inlining leaves\n                  # them behind pointing at definitions that no longer exist —\n                  # and the grammar compiler is handed dangling references. The\n                  # branches keep their const/enum on sourceStrategy, which is\n                  # what actually constrains generation; the mapping is a\n                  # validator convenience and nothing here validates with it.\n                  "discriminator")\n\n\ndef _grammar_safe(node: object) -> object:\n    """Strip keywords the grammar compiler cannot use.\n\n    llama.cpp builds a GBNF grammar from this and never shows it to the model,\n    so descriptions here do not steer anything — the field guidance has to live\n    in DIRECTOR_SYSTEM_PROMPT instead. Dropping them keeps the grammar inside\n    the subset llama.cpp compiles reliably, and roughly halves its size.\n    """\n    if isinstance(node, dict):\n        return {k: _grammar_safe(v) for k, v in node.items() if k not in _GRAMMAR_NOISE}\n    if isinstance(node, list):\n        return [_grammar_safe(v) for v in node]\n    return node\n\n\n@lru_cache(maxsize=2)\ndef _build_json_schema(inline: bool) -> str:\n    schema = _tighten(VideoPlan.model_json_schema())\n    if inline:\n        schema = _grammar_safe(_inline_refs(schema, schema.get("$defs", {})))\n    # Cached as text, so a caller mutating the returned dict cannot poison the\n    # next caller\'s copy.\n    return json.dumps(schema)\n\n\ndef get_json_schema(inline: bool = False) -> dict:\n    """The JSON schema used to constrain generation.\n\n    `inline=True` resolves $defs into the tree and drops validator-only\n    annotations — the form llama.cpp\'s grammar compiler needs.\n\n    Memoised because it cannot change between calls in a running process, so\n    rebuilding it was the same answer computed again. Measured at 26ms per\n    build, which is small — the expensive part is downstream, where the schema\n    is compiled into a GBNF grammar, and that is cached on the engine.\n\n    Returned as a fresh dict each time so a caller mutating it cannot poison\n    the cache.\n    """\n    return json.loads(_build_json_schema(inline))\n'
Path('director/schema.py').write_text(SRC, encoding='utf-8')
print('  wrote director/schema.py', len(SRC), 'bytes')

In [ ]:
# director/prompts.py — generated from the package by build_notebook.py
SRC = '"""System prompts for the AETHER brain.\n\nTwo prompts, because the model does two different jobs. The general one is for\nconversation and open-ended generation; the director one is for producing a\nstoryboard against a grammar. Keeping them apart stops storyboard rules from\nleaking into a chat reply and vice versa.\n"""\n\nAETHER_SYSTEM_PROMPT = (\n    "You are the central intelligence for AETHER, a long-form video production "\n    "studio. You help with scriptwriting, research, structure, SEO and "\n    "storyboarding.\\n"\n    "\\n"\n    "AETHER builds videos from real stock footage (Pixabay, Pexels) and from "\n    "graphics it typesets itself. It does not generate video from a text "\n    "prompt. So when you describe a visual, describe something that could "\n    "actually be found in a stock library or drawn as a chart — never invent "\n    "an asset, a statistic, or an event.\\n"\n    "\\n"\n    "Write plainly. No preamble, no restating the question, no offers of "\n    "further help unless asked."\n)\n\nPLANNING_SYSTEM_PROMPT = (\n    "You are the Visual Director for a long-form video, working out what the "\n    "audience should SEE at each beat of the script.\\n"\n    "\\n"\n    "This is the thinking pass. Nothing you write here is rendered, so reason "\n    "as carefully as the material deserves: read the script for what it is "\n    "actually about, decide the visual approach for the video as a whole, then "\n    "go beat by beat.\\n"\n    "\\n"\n    "For each beat, say which treatment it gets and why that one rather than "\n    "the obvious alternative:\\n"\n    "  chart / map / timeline  - the beat carries numbers, places or dates\\n"\n    "  whiteboard / diagram    - a process, mechanism or labelled structure\\n"\n    "  stickman                - people doing or feeling something\\n"\n    "  editorial_text          - a claim or quote no footage can honestly show\\n"\n    "  stock_text              - a claim wanting atmospheric footage under it\\n"\n    "  stock_video / stock_photo - a real filmed moment: place, texture, mood\\n"\n    "  presenter               - the host addressing the viewer directly\\n"\n    "  t2v / broll             - last resort, when no stock clip could exist\\n"\n    "\\n"\n    "Prefer the simplest visual that explains the idea. A drawn or typeset "\n    "visual renders instantly and identically every run; a searched clip "\n    "depends on what the library happens to hold.\\n"\n    "\\n"\n    "Name the concrete content each beat needs — the actual search phrases, "\n    "the actual numbers for a chart, the actual words for a text card. A "\n    "treatment you cannot fill is the wrong treatment.\\n"\n    "\\n"\n    "Watch the shape of the whole: no more than four consecutive beats sharing "\n    "a treatment, varied shot scale, a change of visual kind every 30-40 "\n    "seconds, and the host used sparingly. Note any contradiction in place, "\n    "time of day or weather between consecutive beats.\\n"\n    "\\n"\n    "End with a compact beat-by-beat list. The next pass turns it into JSON "\n    "verbatim, so decide everything here."\n)\n\nDIRECTOR_SYSTEM_PROMPT = (\n    "You are the Visual Director for a long-form video. For every beat of the "\n    "script you decide what the audience SEES at that moment, and why that "\n    "choice serves the story.\\n"\n    "\\n"\n    "Your answer is consumed by software, not by a person. It must match the "\n    "supplied JSON schema exactly. No markdown, no commentary, no code fence.\\n"\n    "\\n"\n    "ALWAYS PREFER THE SIMPLEST VISUAL THAT EXPLAINS THE IDEA. A drawn or "\n    "typeset visual renders instantly, looks identical on every run, and stays "\n    "editable. A searched clip depends on what the library happens to hold. "\n    "Choose in this order, and only move down when the option above genuinely "\n    "cannot communicate the point:\\n"\n    "  1. chart / map / timeline  - the beat carries numbers, places or dates\\n"\n    "  2. whiteboard / diagram    - a process, mechanism or labelled structure\\n"\n    "  3. stickman                - people doing or feeling something\\n"\n    "  4. editorial_text          - a claim or quote no footage can honestly show\\n"\n    "  5. stock_text              - a claim wanting atmospheric footage under it\\n"\n    "  6. stock_video / stock_photo - a real filmed moment: place, texture, mood\\n"\n    "  7. presenter               - the host addressing the viewer directly\\n"\n    "  8. t2v / broll             - last resort, when no stock clip could exist\\n"\n    "\\n"\n    "THE ONE RULE THAT BREAKS EVERYTHING IF YOU IGNORE IT\\n"\n    "Choosing a visual type is a promise to supply what that type draws from. "\n    "The renderer has nothing else to fall back on — a type without its "\n    "payload comes out as a blank frame:\\n"\n    "  stock_video, stock_photo, stock_text -> stockRequirements.queries\\n"\n    "  stock_text, editorial_text           -> textOverlay.text\\n"\n    "  stickman, whiteboard, chart, map, timeline, diagram -> graphic.items\\n"\n    "  t2v, broll, presenter                -> nothing extra\\n"\n    "If you cannot fill the payload for a type, you have picked the wrong "\n    "type. Pick one you can fill.\\n"\n    "\\n"\n    "FIELD RULES\\n"\n    "- index: the zero-based position of the beat, copied from the cue list. "\n    "Every beat needs one, and they must be unique and in order.\\n"\n    "- stockRequirements: required for stock_video, stock_photo and stock_text. "\n    "Write queries the way a stock photographer searches — subject, action, "\n    "setting. \'expensive grocery shopping\' finds footage; \'inflation erodes "\n    "purchasing power\' finds nothing.\\n"\n    "- textOverlay: required for stock_text and editorial_text. Under 12 words. "\n    "It is a caption, not the narration repeated.\\n"\n    "- graphic.items: required for stickman, whiteboard, chart, map, timeline "\n    "and diagram, and it must carry real content pulled from the narration — "\n    "\'Label: Number\' for chart, \'Date: Event\' for timeline, place names for "\n    "map, steps for whiteboard, \'action:expression\' for stickman. A graphic "\n    "with no items renders as a blank card.\\n"\n    "- Only claim a chart, map or timeline when the narration actually contains "\n    "the numbers, places or dates to fill it. Otherwise choose footage.\\n"\n    "\\n"\n    "WHERE THE FOOTAGE COMES FROM\\n"\n    "Three libraries, and they are not interchangeable:\\n"\n    "  pexels, pixabay  - modern stock. What the world looks like now: people, "\n    "offices, streets, nature, technology in use today.\\n"\n    "  archive_org      - a film archive. Newsreels, government and industrial "\n    "films, old television, documentaries, period footage.\\n"\n    "\\n"\n    "Use archive_org when the beat is ABOUT a time that has passed and "\n    "authentic film of it says more than a modern re-creation ever could — a "\n    "war, a moon landing, a protest, a crash, a decade, an obsolete machine, a "\n    "historical figure, a vanished way of working. A 1969 launch filmed in 1969 "\n    "carries weight that no modern stock clip can borrow.\\n"\n    "\\n"\n    "Do NOT reach for it out of habit. A beat about someone checking their "\n    "phone in a coffee shop, a modern office, today\'s supermarket, current "\n    "technology — that is modern stock, and archive footage there is simply "\n    "the wrong answer. \'Americans shopping today\' is pexels. \'Americans "\n    "queuing for petrol in 1973\' is archive_org.\\n"\n    "\\n"\n    "Every stock beat MUST answer both: sourceStrategy is \'archival\' or "\n    "\'modern_stock\' when the beat clearly wants one, and \'auto\' only when "\n    "either would genuinely serve — not as a way to avoid choosing. "\n    "preferredSources names at least one library, best first, each listed "\n    "once. An empty list is not \'let AETHER decide\', it is a beat with no "\n    "source at all, and it ends up searched the same generic way as every "\n    "other beat.\\n"\n    "\\n"\n    "ARCHIVE SEARCHES ARE WRITTEN DIFFERENTLY\\n"\n    "A stock library is catalogued by what a clip SHOWS. A film archive is "\n    "catalogued by what a film IS — its title, its kind, its year, who made "\n    "it. So archiveQueries name the artefact, not the picture:\\n"\n    "  \'happy worker in factory\'        -> finds nothing in an archive\\n"\n    "  \'1940s wartime factory newsreel\' -> finds the film\\n"\n    "  \'early computers\'                -> too vague\\n"\n    "  \'1960s mainframe computer documentary\' -> finds the film\\n"\n    "Only fill archiveQueries in when archive_org is in preferredSources.\\n"\n    "\\n"\n    "Set timePeriod only when the script actually establishes the date. A "\n    "guessed decade points the search at the wrong footage, which is worse "\n    "than leaving it open. If the narration does not say when, leave it out.\\n"\n    "\\n"\n    "AN ARCHIVAL BEAT CARRIES MORE THAN A MODERN ONE\\n"\n    "Choosing \'archival\' commits you to four more answers, and the schema will "\n    "not let you leave them out:\\n"\n    "  archiveQueries   - what to ask the archive for, named as artefacts\\n"\n    "  excerpt          - an archive item is a WHOLE FILM, so say how many "\n    "seconds you want and, in selectionIntent, what should be visible in them: "\n    "\'workers operating wartime machinery\', not \'wartime footage\'\\n"\n    "  editorialPurpose - why this footage earns its place in the story\\n"\n    "  timePeriod       - when the narration establishes a date\\n"\n    "None of these apply to a modern beat and none of them are offered there. "\n    "If you cannot answer them, the beat is not really archival — say "\n    "modern_stock and search a stock library instead.\\n"\n    "\\n"\n    "Even a historical documentary should not be archive footage end to end. "\n    "Cut between period film, a chart of the numbers, a modern shot of the "\n    "consequences, a map, a quote card. Ten archival clips in a row is a "\n    "slideshow, not a documentary.\\n"\n    "\\n"\n    "PACING\\n"\n    "- Never let more than four consecutive beats share a visualType.\\n"\n    "- Vary shot scale; back-to-back identical framing kills momentum.\\n"\n    "- Change the kind of visual every 30-40 seconds.\\n"\n    "- Use presenter sparingly: the hook, a section turn, the close.\\n"\n    "\\n"\n    "Put any contradiction in place, time of day or weather between "\n    "consecutive beats into `warnings`."\n)\n'
Path('director/prompts.py').write_text(SRC, encoding='utf-8')
print('  wrote director/prompts.py', len(SRC), 'bytes')

In [ ]:
# director/cache.py — generated from the package by build_notebook.py
SRC = '"""Disk cache for director results.\n\nA storyboard costs minutes of GPU on a T4, and re-running the same script\nduring development is the common case. Keyed on everything that changes the\nanswer, so a schema or model change misses rather than serving a stale shape.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport tempfile\nfrom typing import Any, Optional\n\nCACHE_DIR = os.environ.get("DIRECTOR_CACHE_DIR", "/tmp/director_cache")\n\n\ndef init_cache() -> None:\n    os.makedirs(CACHE_DIR, exist_ok=True)\n\n\ndef get_cache_key(script: str, style: str, model_id: str, schema_version: str, extra: str = "") -> str:\n    payload = "|".join([script, style, model_id, schema_version, extra])\n    return hashlib.sha256(payload.encode("utf-8")).hexdigest()\n\n\ndef _path(cache_key: str) -> str:\n    return os.path.join(CACHE_DIR, f"{cache_key}.json")\n\n\ndef get_cached_result(cache_key: str) -> Optional[Any]:\n    try:\n        with open(_path(cache_key), encoding="utf-8") as handle:\n            return json.load(handle)\n    except (FileNotFoundError, json.JSONDecodeError):\n        # A half-written file from an interrupted run should miss, not crash.\n        return None\n\n\ndef set_cached_result(cache_key: str, data: Any) -> None:\n    os.makedirs(CACHE_DIR, exist_ok=True)\n    # Write-then-rename so a reader never sees a partial file.\n    fd, tmp = tempfile.mkstemp(dir=CACHE_DIR, suffix=".tmp")\n    try:\n        with os.fdopen(fd, "w", encoding="utf-8") as handle:\n            json.dump(data, handle)\n        os.replace(tmp, _path(cache_key))\n    except Exception:\n        if os.path.exists(tmp):\n            os.unlink(tmp)\n        raise\n'
Path('director/cache.py').write_text(SRC, encoding='utf-8')
print('  wrote director/cache.py', len(SRC), 'bytes')

In [ ]:
# director/inference.py — generated from the package by build_notebook.py
SRC = '"""Inference through llama.cpp.\n\nThe model is Qwen3.8-27B as a GGUF quantization, loaded in-process by\nllama-cpp-python rather than served by a separate vLLM process. That choice is\nforced by the hardware: Kaggle\'s T4s are Turing (sm_75), and the\ncompressed-tensors 4-bit build of this model decodes through Marlin kernels\nthat need sm_80. llama.cpp\'s CUDA kernels run on Turing, and GGUF is a\ndifferent quantization format entirely.\n\nTwo consequences worth knowing:\n\n  * llama.cpp splits a model across GPUs by layer, not by tensor. A second\n    card buys capacity, not speed — the cards take turns.\n  * There is one model in one process, and it is not thread-safe. Requests are\n    serialized behind a lock rather than batched.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nimport re\nimport threading\nimport time\nfrom typing import Any, Dict, Iterator, List, Optional, Tuple\n\nfrom .prompts import (\n    AETHER_SYSTEM_PROMPT,\n    DIRECTOR_SYSTEM_PROMPT,\n    PLANNING_SYSTEM_PROMPT,\n)\nfrom .schema import (\n    NEEDS_GRAPHIC,\n    NEEDS_STOCK,\n    NEEDS_TEXT,\n    get_json_schema,\n    plan_violations,\n)\n\n_THINK_BLOCK = re.compile(r"<think>.*?</think>", re.DOTALL)\n\n\ndef strip_thinking(text: str, started_inside: bool = False) -> str:\n    """Drop the reasoning and return the answer.\n\n    The opening <think> usually is NOT in the output. This model\'s chat\n    template ends the prompt with a bare `<think>\\\\n`, so generation begins\n    already inside the block and only the closing tag is ever generated. A\n    regex looking for a matched pair therefore finds nothing and the caller\n    gets several hundred tokens of deliberation where it expected a title.\n\n    Everything up to the last closing tag is deliberation; what follows is the\n    answer. An unclosed block means generation stopped mid-thought, and there\n    is no answer in it at all — an empty string is the honest result.\n\n    `started_inside` says the prompt opened the block, which the caller knows\n    and this function cannot: with reasoning enabled, text carrying no tags at\n    all is not an answer, it is deliberation that never closed. Returning it\n    put a model thinking aloud — "We need answer user\'s request. Need produce\n    final only spoken narration… Let\'s count." — into a finished script.\n    """\n    text = text or ""\n    if "</think>" in text:\n        return text.rsplit("</think>", 1)[-1].strip()\n    text = _THINK_BLOCK.sub("", text)\n    if "<think>" in text:\n        return text.split("<think>", 1)[0].strip()\n    if started_inside:\n        return ""\n    return text.strip()\n\n\ndef preload_cuda_libraries() -> List[str]:\n    """Load the CUDA runtime into the process before llama.cpp asks for it.\n\n    The prebuilt llama-cpp-python wheels link against libcudart.so.12, which is\n    not on the default loader path in these images — importing llama_cpp fails\n    with "libcudart.so.12: cannot open shared object file" even though CUDA is\n    installed and working. Setting LD_LIBRARY_PATH from inside Python is too\n    late, because the linker read it at process start.\n\n    Opening the libraries here with RTLD_GLOBAL puts their symbols in the\n    global namespace, so the later dlopen of libllama.so resolves against\n    them. Call this before `import llama_cpp`.\n    """\n    import ctypes\n    import glob\n    import site\n\n    roots: List[str] = []\n    try:\n        roots.extend(site.getsitepackages())\n    except Exception:\n        pass\n    roots.append(os.path.dirname(os.path.dirname(os.__file__)))\n\n    loaded: List[str] = []\n    # Order matters: cublas depends on cublasLt, and both depend on cudart.\n    for soname in ("libcudart.so.12", "libcublasLt.so.12", "libcublas.so.12"):\n        if _already_loaded(soname):\n            continue\n        for root in roots:\n            hits = glob.glob(os.path.join(root, "nvidia", "*", "lib", soname))\n            hits += glob.glob(os.path.join(root, "torch", "lib", soname))\n            if not hits:\n                continue\n            try:\n                ctypes.CDLL(hits[0], mode=ctypes.RTLD_GLOBAL)\n                loaded.append(hits[0])\n                break\n            except OSError:\n                continue\n    return loaded\n\n\ndef _already_loaded(soname: str) -> bool:\n    import ctypes\n\n    try:\n        ctypes.CDLL(soname, mode=ctypes.RTLD_GLOBAL)\n        return True\n    except OSError:\n        return False\n\n\ndef quiet_backend_logs() -> bool:\n    """Silence llama.cpp\'s per-token logging once the model is loaded.\n\n    The CUDA backend prints "CUDA Graph id N reused" for every decoded token.\n    During the 2-5 minute load that chatter is the only sign of progress and is\n    worth keeping, but a grammar-constrained plan is thousands of tokens and the\n    same line thousands of times does real damage in a notebook: measured on\n    Kaggle it pushed the output past 370 KB, which is enough for the editor to\n    start answering "Failed to save draft", and it buries the one line that\n    actually says what happened.\n\n    Returns False if the installed llama-cpp-python does not expose the log\n    hook, because a noisy notebook is a far better outcome than a stage that\n    dies trying to make it quiet.\n    """\n    try:\n        import ctypes\n\n        import llama_cpp\n\n        # Held on the module so the callback is not garbage collected while C\n        # still holds the pointer — that crashes the kernel rather than logging.\n        @ctypes.CFUNCTYPE(None, ctypes.c_int, ctypes.c_char_p, ctypes.c_void_p)\n        def _swallow(level, text, user_data):  # noqa: ANN001, ARG001\n            return None\n\n        globals()["_LOG_SINK"] = _swallow\n        llama_cpp.llama_log_set(_swallow, ctypes.c_void_p(0))\n        return True\n    except Exception:  # noqa: BLE001\n        return False\n\n\nclass DirectorInference:\n    """Wraps the model with the three shapes AETHER asks for: chat, generate, plan."""\n\n    def __init__(\n        self,\n        model_path: str,\n        n_ctx: int = 16384,\n        n_gpu_layers: int = -1,\n        verbose: bool = False,\n        **llama_kwargs: Any,\n    ):\n        preload_cuda_libraries()\n        from llama_cpp import Llama  # imported late so the preload runs first\n\n        if not os.path.exists(model_path):\n            raise FileNotFoundError(f"GGUF not found at {model_path}")\n\n        self.model_path = model_path\n        self.n_ctx = n_ctx\n        self.llm = Llama(\n            model_path=model_path,\n            n_gpu_layers=n_gpu_layers,  # -1 offloads every layer it can\n            n_ctx=n_ctx,\n            verbose=verbose,\n            **llama_kwargs,\n        )\n        # One model, one process, no batching. Without this, two concurrent\n        # requests interleave into the same context and corrupt each other.\n        self._lock = threading.Lock()\n        self._formatter = self._build_formatter()\n\n    def _build_formatter(self):\n        """A renderer for the model\'s own chat template that we can pass flags to.\n\n        This model reasons by default, at the template\'s \'xhigh\' effort, and\n        the only way to turn that off is the `enable_thinking` template\n        variable. create_chat_completion() does not forward unknown keywords to\n        the template — it passes a fixed argument list — so the flag cannot get\n        there through the normal API. Rendering the prompt ourselves and\n        calling create_completion() is the way in.\n\n        Returns None if this llama-cpp-python does not expose what we need, in\n        which case everything still works through the chat API; the model just\n        deliberates first and pays for it in tokens.\n        """\n        try:\n            from llama_cpp.llama_chat_format import Jinja2ChatFormatter\n\n            template = (self.llm.metadata or {}).get("tokenizer.chat_template")\n            if not template:\n                return None\n            return Jinja2ChatFormatter(\n                template=template,\n                eos_token=self.llm._model.token_get_text(self.llm.token_eos()),\n                bos_token=self.llm._model.token_get_text(self.llm.token_bos()),\n                add_generation_prompt=True,\n            )\n        except Exception as exc:  # noqa: BLE001\n            print(f"[director] chat template not renderable directly ({exc}); "\n                  "reasoning cannot be disabled and replies will be slower.")\n            return None\n\n    def _render(self, messages: List[Dict[str, str]], thinking: bool,\n                effort: str = "xhigh") -> Optional[str]:\n        """The prompt string, with reasoning explicitly on or off."""\n        if self._formatter is None:\n            return None\n        try:\n            rendered = self._formatter(\n                llama=self.llm,\n                messages=messages,\n                enable_thinking=thinking,\n                # The template raises on anything outside its own list, so\n                # never hand it a value straight from a request body.\n                reasoning_effort=effort if effort in self.EFFORTS else "xhigh",\n            )\n            return rendered.prompt\n        except Exception:\n            # A template that rejects our flags is not worth failing over.\n            self._formatter = None\n            return None\n\n    # -- helpers ----------------------------------------------------------\n\n    @staticmethod\n    def _with_system(messages: List[Dict[str, str]], system: str) -> List[Dict[str, str]]:\n        """Return a NEW list with a system turn in front — never mutate the caller\'s."""\n        if any(m.get("role") == "system" for m in messages):\n            return list(messages)\n        return [{"role": "system", "content": system}, *messages]\n\n    #: The template validates this and raises on anything else.\n    EFFORTS = ("low", "medium", "xhigh")\n\n    # -- conversational ---------------------------------------------------\n\n    def chat(\n        self,\n        messages: List[Dict[str, str]],\n        temperature: float = 0.7,\n        max_tokens: int = 4096,\n        thinking: bool = True,\n        reasoning_effort: str = "xhigh",\n        stream: bool = False,\n        **kwargs: Any,\n    ):\n        """A normal chat turn. Returns text, or a delta iterator when streaming.\n\n        Reasoning is ON. It costs tokens — at roughly 10 tokens/second a long\n        deliberation is minutes before the answer starts — but scriptwriting,\n        research and fact-checking are exactly the work it pays for. Callers\n        that want speed over depth pass thinking=False or a lower effort.\n        """\n        prepared = self._with_system(messages, AETHER_SYSTEM_PROMPT)\n        prompt = self._render(prepared, thinking, reasoning_effort)\n\n        if not stream:\n            with self._lock:\n                if prompt is not None:\n                    result = self.llm.create_completion(\n                        prompt=prompt, temperature=temperature,\n                        max_tokens=max_tokens, **kwargs,\n                    )\n                    raw = result["choices"][0]["text"] or ""\n                else:\n                    result = self.llm.create_chat_completion(\n                        messages=prepared, temperature=temperature,\n                        max_tokens=max_tokens, **kwargs,\n                    )\n                    raw = result["choices"][0]["message"]["content"] or ""\n\n            answer = strip_thinking(raw, started_inside=bool(thinking))\n            if not answer and raw.strip():\n                # Everything generated was deliberation that never closed, so\n                # there is no answer to return. Saying so beats an empty string\n                # that looks like the model had nothing to say.\n                raise ValueError(\n                    f"The model spent all {max_tokens} tokens reasoning and never "\n                    "reached an answer. Raise max_tokens, or disable reasoning."\n                )\n            return answer\n\n        return self._stream(prepared, prompt, temperature, max_tokens, kwargs, thinking)\n\n    def _stream(self, messages, prompt, temperature, max_tokens, kwargs,\n                thinking: bool = True) -> Iterator[str]:\n        """Yield content deltas, holding the model lock for the whole stream."""\n        with self._lock:\n            if prompt is not None:\n                chunks = self.llm.create_completion(\n                    prompt=prompt, temperature=temperature,\n                    max_tokens=max_tokens, stream=True, **kwargs,\n                )\n                deltas = (c["choices"][0].get("text") or "" for c in chunks)\n            else:\n                chunks = self.llm.create_chat_completion(\n                    messages=messages, temperature=temperature,\n                    max_tokens=max_tokens, stream=True, **kwargs,\n                )\n                deltas = (c["choices"][0].get("delta", {}).get("content") or "" for c in chunks)\n\n            # Generation begins already inside a reasoning block whenever the\n            # prompt opened one, which is exactly what the chat template does\n            # with thinking on: it ends on a bare `<think>` and only the CLOSING\n            # tag is ever generated.\n            #\n            # This started at False — the opposite of the line above it — so\n            # every reasoning token was streamed straight to the caller until a\n            # closing tag arrived, and if the model never emitted one, the whole\n            # deliberation was the answer. That is how a script came back\n            # beginning "We need answer user\'s request… Let\'s count."\n            #\n            # With thinking off the template pre-closes the block, so generation\n            # starts outside it and nothing should be suppressed.\n            in_thought = bool(thinking)\n            seen_close = False\n            for delta in deltas:\n                if not delta:\n                    continue\n                if not seen_close and "</think>" in delta:\n                    seen_close = True\n                    in_thought = False\n                    delta = delta.split("</think>", 1)[-1]\n                    if not delta:\n                        continue\n                # Reasoning arrives token by token, so it cannot be regexed out\n                # after the fact — it has to be gated as it goes past.\n                if "<think>" in delta:\n                    in_thought = True\n                    delta = delta.split("<think>")[0]\n                if in_thought:\n                    if "</think>" not in delta:\n                        continue\n                    in_thought = False\n                    delta = delta.split("</think>")[-1]\n                if delta:\n                    yield delta\n\n    def generate(\n        self,\n        prompt: str,\n        temperature: float = 0.7,\n        max_tokens: int = 4096,\n        thinking: bool = True,\n        reasoning_effort: str = "xhigh",\n        response_format: Optional[Dict[str, Any]] = None,\n        **kwargs: Any,\n    ) -> str:\n        """One-shot completion for the structured application tasks."""\n        if response_format is not None:\n            kwargs["response_format"] = response_format\n        return self.chat(\n            [{"role": "user", "content": prompt}],\n            temperature=temperature,\n            max_tokens=max_tokens,\n            thinking=thinking,\n            reasoning_effort=reasoning_effort,\n            **kwargs,\n        )\n\n    # -- storyboard -------------------------------------------------------\n\n    @staticmethod\n    def _repair(plan: dict, cues: Optional[List[Dict[str, Any]]]) -> List[str]:\n        """Last resort: make each broken beat renderable, in place.\n\n        Only runs when the model has already been asked to fix its own work and\n        did not. Rather than fail the whole storyboard over one beat, downgrade\n        the beat to something the data it *did* supply can actually render —\n        and where nothing was supplied, fall back to the narration itself.\n        """\n        narration = {}\n        for position, cue in enumerate(cues or []):\n            key = cue.get("index", position)\n            narration[key] = str(cue.get("text") or cue.get("subtitle") or "").strip()\n\n        notes: List[str] = []\n        for position, scene in enumerate(plan.get("scenes") or []):\n            # Read without a default: scene.get("index", position) would hand\n            # back a perfectly valid int for a key that is not there, and the\n            # check below would then decide nothing was wrong.\n            index = scene.get("index")\n            if not isinstance(index, int) or isinstance(index, bool):\n                scene["index"] = index = position\n                notes.append(f"scene {position}: supplied a missing index")\n\n            visual = scene.get("visualType")\n            line = narration.get(index, "")\n\n            if visual in NEEDS_TEXT and not ((scene.get("textOverlay") or {}).get("text") or "").strip():\n                words = line.split()\n                if words:\n                    scene["textOverlay"] = {\n                        "text": " ".join(words[:12]),\n                        "emphasis": "",\n                        "style": "emphasis",\n                    }\n                    notes.append(f"scene {index}: captioned {visual} from its own narration")\n                else:\n                    scene["visualType"] = "broll"\n                    notes.append(f"scene {index}: {visual} had no text and no narration, demoted to broll")\n\n            if visual in NEEDS_GRAPHIC and not ((scene.get("graphic") or {}).get("items")):\n                # A blank chart is worse than honest footage.\n                scene["visualType"] = "broll"\n                notes.append(f"scene {index}: {visual} had no items to draw, demoted to broll")\n\n            if visual in NEEDS_STOCK and not ((scene.get("stockRequirements") or {}).get("queries")):\n                scene["visualType"] = "broll"\n                notes.append(f"scene {index}: {visual} had no queries, demoted to broll")\n\n        if notes:\n            plan.setdefault("warnings", []).extend(notes)\n        return notes\n\n    def generate_plan(\n        self,\n        script: str,\n        style: str = "documentary",\n        title: str = "Untitled",\n        cues: Optional[List[Dict[str, Any]]] = None,\n        brief: str = "",\n        max_tokens: int = 6144,\n        reasoning_effort: str = "xhigh",\n        plan_tokens: int = 4096,\n    ) -> Tuple[dict, float]:\n        """Produce a video plan in AETHER\'s `parseVideoPlan` shape.\n\n        Done in two passes, because a grammar and a reasoning block cannot\n        share one call. The schema compiles to a GBNF grammar that forces the\n        very first token to be an opening brace, so a model that wants to\n        think first has nowhere to put the thought — a single constrained call\n        gets structure at the cost of any deliberation at all.\n\n        So the thinking happens first, unconstrained, where the model can\n        weigh what each beat is about and what treatment earns its place. The\n        second pass is transcription: the same decisions, now under the\n        grammar. Set reasoning_effort=None to skip the first pass when speed\n        matters more than the choice being any good.\n        """\n        beats = ""\n        if cues:\n            lines = []\n            for position, cue in enumerate(cues):\n                text = str(cue.get("text") or cue.get("subtitle") or "").strip()\n                # AETHER merges the answer back onto its own scenes by index,\n                # so its numbering wins whenever it sends one.\n                index = cue.get("index", position)\n                start = cue.get("start", cue.get("timestamp"))\n                stamp = f" (t={start}s)" if start is not None else ""\n                lines.append(f"  index {index}{stamp}: {text}")\n            beats = (\n                "\\n\\nBEATS — return exactly one scene for each line below, "\n                "reusing these index numbers exactly:\\n" + "\\n".join(lines)\n            )\n\n        request = (\n            f"Title: {title}\\n"\n            f"Visual style: {style}\\n"\n            + (f"\\n{brief}\\n" if brief else "")\n            + f"\\nScript:\\n{script}"\n            f"{beats}"\n        )\n\n        from .schema import VideoPlan\n\n        started = time.time()\n        plan: dict = {}\n\n        # Pass one: think. Unconstrained, so the reasoning block is allowed to\n        # exist at all.\n        reasoning = ""\n        if reasoning_effort:\n            reasoning = self.chat(\n                [{"role": "system", "content": PLANNING_SYSTEM_PROMPT},\n                 {"role": "user", "content": request}],\n                temperature=0.3,\n                max_tokens=plan_tokens,\n                thinking=True,\n                reasoning_effort=reasoning_effort,\n            )\n\n        messages = [\n            {"role": "system", "content": DIRECTOR_SYSTEM_PROMPT},\n            {"role": "user", "content": request},\n        ]\n        if reasoning:\n            # Pass two is transcription, not a second opinion. Handing the\n            # decisions back as the assistant\'s own words is what stops it\n            # re-deciding them under the grammar.\n            messages += [\n                {"role": "assistant", "content": reasoning},\n                {"role": "user", "content":\n                    "Now emit exactly that plan as JSON matching the schema. "\n                    "Keep every decision you just made — same treatments, same "\n                    "queries, same numbers, same wording. Change nothing."},\n            ]\n\n        # The grammar guarantees the shape but not the sense: it cannot tie a\n        # visualType to the payload that type needs, so the model can pick\n        # \'editorial_text\' and leave textOverlay null. Showing it the specific\n        # broken beats and asking again fixes this far more often than not.\n        for attempt in range(self.PLAN_ATTEMPTS):\n            plan = self._plan_once(messages, max_tokens, started)\n            problems = plan_violations(plan)\n            if not problems:\n                VideoPlan(**plan)\n                return plan, time.time() - started\n\n            if attempt == self.PLAN_ATTEMPTS - 1:\n                break\n\n            listing = "\\n".join(f"  - scene index {i}: {msg}" for i, msg in problems)\n            messages = messages + [\n                {"role": "assistant", "content": json.dumps(plan)},\n                {\n                    "role": "user",\n                    "content": (\n                        "That plan is not renderable. Every one of these beats "\n                        "chose a visual type without supplying what that type "\n                        "needs to draw:\\n" + listing + "\\n\\n"\n                        "Return the whole plan again with those beats fixed. "\n                        "Either fill in the missing payload, or change the beat "\n                        "to a visual type whose payload you actually have. "\n                        "Leave every other beat as it was."\n                    ),\n                },\n            ]\n\n        # Still broken after asking. Repair rather than lose the storyboard —\n        # one blank beat is not worth failing the other thirty.\n        self._repair(plan, cues)\n        VideoPlan(**plan)\n        return plan, time.time() - started\n\n    #: How many times to ask before repairing the answer ourselves.\n    PLAN_ATTEMPTS = 2\n\n    def _plan_grammar(self):\n        """The compiled GBNF grammar, built once per process.\n\n        Passing `response_format` makes llama-cpp-python compile the schema to a\n        grammar on every single call, and for this schema that dominates the\n        request: a one-beat plan with the reasoning pass disabled took 286s\n        against roughly 30s of actual generation. Compiling once turns a\n        per-request cost into a one-off, which is the difference between fitting\n        inside a 300s gateway timeout and never fitting.\n\n        Falls back to response_format if this build of llama-cpp-python does not\n        expose LlamaGrammar — slow beats broken.\n        """\n        if getattr(self, "_grammar_cache", None) is not None:\n            return self._grammar_cache\n        try:\n            from llama_cpp import LlamaGrammar\n\n            self._grammar_cache = LlamaGrammar.from_json_schema(\n                json.dumps(get_json_schema(inline=True)), verbose=False\n            )\n        except Exception:  # noqa: BLE001\n            self._grammar_cache = False\n        return self._grammar_cache\n\n    def _plan_once(self, messages, max_tokens: int, started: float) -> dict:\n        grammar = self._plan_grammar()\n        constraint = (\n            {"grammar": grammar}\n            if grammar\n            else {"response_format": {"type": "json_object", "schema": get_json_schema(inline=True)}}\n        )\n        with self._lock:\n            result = self.llm.create_chat_completion(\n                messages=messages,\n                temperature=0.0,\n                max_tokens=max_tokens,\n                **constraint,\n            )\n\n        raw = result["choices"][0]["message"]["content"] or ""\n        cleaned = strip_thinking(raw)\n        try:\n            return json.loads(cleaned)\n        except json.JSONDecodeError as exc:\n            # The grammar should make this unreachable; hitting max_tokens\n            # mid-object is the one way it happens.\n            raise ValueError(\n                f"Model returned invalid JSON after {time.time()-started:.1f}s ({exc}). "\n                f"Finish reason: {result[\'choices\'][0].get(\'finish_reason\')}. "\n                f"Tail: ...{cleaned[-400:]!r}"\n            ) from exc\n\n    # Older callers use this name.\n    def generate_storyboard(self, script: str, style: str = "documentary", **kwargs):\n        return self.generate_plan(script=script, style=style, **kwargs)\n'
Path('director/inference.py').write_text(SRC, encoding='utf-8')
print('  wrote director/inference.py', len(SRC), 'bytes')

In [ ]:
# director/api.py — generated from the package by build_notebook.py
SRC = '"""FastAPI surface for AETHER.\n\nThree ways in, one model behind them: /chat for conversation, /generate for the\nstructured application tasks, /director for a storyboard against the grammar.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nimport threading\nfrom contextlib import asynccontextmanager\nfrom typing import Any, Dict, List, Optional\n\nfrom fastapi import Depends, FastAPI, HTTPException\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom fastapi.responses import StreamingResponse\nfrom fastapi.security import HTTPAuthorizationCredentials, HTTPBearer\nfrom pydantic import BaseModel\n\nfrom .cache import get_cache_key, get_cached_result, init_cache, set_cached_result\nfrom .inference import DirectorInference\n\nAPI_KEY = os.environ.get("DIRECTOR_API_KEY", "test-key-change-me")\n# The notebook exports all of these before uvicorn starts. The defaults only\n# matter when running the API standalone.\nMODEL_ID = os.environ.get("DIRECTOR_MODEL", "unsloth/Qwen3.8-27B-GGUF:Q5_K_M")\nMODEL_PATH = os.environ.get("DIRECTOR_MODEL_PATH", "")\nN_CTX = int(os.environ.get("DIRECTOR_N_CTX", "16384"))\n# Bumped when the schema changes shape in a way a caller can observe.\n#   2.1  made sourceStrategy and preferredSources required\n#   2.2  split stockRequirements on sourceStrategy, so an archival beat must\n#        carry archiveQueries, excerpt and editorialPurpose, and a modern beat\n#        is not offered them at all\n#\n# This is also part of the plan cache key, which matters more than it looks: a\n# session still serving 2.0 would otherwise hand back cached plans missing the\n# very fields the new schema exists to force. And /health reports it, so a\n# caller can tell which generation it is actually talking to instead of assuming\n# the redeploy landed.\nSCHEMA_VERSION = "2.2-aether"\n\n_engine: Optional[DirectorInference] = None\n\n\ndef get_engine() -> DirectorInference:\n    """Load on first use.\n\n    Loading is minutes of work and gigabytes of VRAM, so it must not happen\n    during import — uvicorn would appear to hang, and /health would be\n    unreachable exactly when someone is trying to find out what is wrong.\n    """\n    global _engine\n    if _engine is None:\n        if not MODEL_PATH:\n            raise HTTPException(503, "DIRECTOR_MODEL_PATH is not set — no GGUF to load.")\n        _engine = DirectorInference(model_path=MODEL_PATH, n_ctx=N_CTX)\n    return _engine\n\n\n@asynccontextmanager\nasync def lifespan(_: FastAPI):\n    init_cache()\n    yield\n\n\napp = FastAPI(title="AETHER Qwen Brain", lifespan=lifespan)\n\n# The browser never reaches this directly — AETHER\'s node server proxies it —\n# but allowing the origin makes a direct curl or a tunnel probe behave.\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=["*"],\n    allow_methods=["*"],\n    allow_headers=["*"],\n)\n\nsecurity = HTTPBearer(auto_error=True)\n\n\ndef check_token(creds: HTTPAuthorizationCredentials = Depends(security)) -> str:\n    if creds.credentials != API_KEY:\n        raise HTTPException(401, "Invalid API key")\n    return creds.credentials\n\n\nclass ChatRequest(BaseModel):\n    messages: List[Dict[str, str]]\n    temperature: float = 0.7\n    # Generous, because on the fallback path the model reasons first and a\n    # tight budget is spent entirely on deliberation, returning no answer.\n    max_tokens: int = 4096\n    thinking: bool = True\n    reasoning_effort: str = "xhigh"\n    stream: bool = False\n\n\nclass GenerateRequest(BaseModel):\n    prompt: str\n    temperature: float = 0.7\n    max_tokens: int = 4096\n    thinking: bool = True\n    reasoning_effort: str = "xhigh"\n    response_format: Optional[Dict[str, Any]] = None\n\n\nclass DirectorRequest(BaseModel):\n    script: str\n    title: str = "Untitled"\n    style: str = "documentary"\n    language: str = "en"\n    cues: Optional[List[Dict[str, Any]]] = None\n    brief: str = ""\n    max_tokens: int = 6144\n    # The thinking pass. None skips it and goes straight to the grammar, which\n    # is faster and noticeably less considered.\n    reasoning_effort: Optional[str] = "xhigh"\n    plan_tokens: int = 4096\n    no_cache: bool = False\n\n\n# /health is deliberately unauthenticated: AETHER polls it every few seconds to\n# decide whether Qwen is up, and a health probe that can fail on auth would\n# report the model down whenever the key is merely misconfigured.\n@app.get("/health")\ndef health() -> dict:\n    # `loaded` distinguishes "the server is up" from "the model is in VRAM".\n    # AETHER only needs the former to stop falling back to NIM, but the\n    # difference is the whole answer when a first request seems to hang.\n    return {\n        "status": "ok",\n        "model": MODEL_ID,\n        "loaded": _engine is not None,\n        "schema_version": SCHEMA_VERSION,\n    }\n\n\n@app.get("/model")\ndef model_info() -> dict:\n    return {\n        "model": MODEL_ID,\n        "path": MODEL_PATH,\n        "n_ctx": N_CTX,\n        "loaded": _engine is not None,\n        "schema_version": SCHEMA_VERSION,\n    }\n\n\n@app.post("/chat")\ndef chat_endpoint(req: ChatRequest, _: str = Depends(check_token)):\n    engine = get_engine()\n\n    if not req.stream:\n        try:\n            text = engine.chat(\n                req.messages,\n                temperature=req.temperature,\n                max_tokens=req.max_tokens,\n                thinking=req.thinking,\n                reasoning_effort=req.reasoning_effort,\n            )\n        except Exception as exc:  # noqa: BLE001\n            raise HTTPException(502, f"Model error: {exc}") from exc\n        return {"choices": [{"message": {"role": "assistant", "content": text}}]}\n\n    try:\n        chunks = engine.chat(\n            req.messages,\n            temperature=req.temperature,\n            max_tokens=req.max_tokens,\n            thinking=req.thinking,\n            reasoning_effort=req.reasoning_effort,\n            stream=True,\n        )\n    except Exception as exc:  # noqa: BLE001\n        raise HTTPException(502, f"Model error: {exc}") from exc\n\n    def sse():\n        try:\n            # DirectorInference._stream yields content deltas as plain strings,\n            # with any reasoning already gated out.\n            for delta in chunks:\n                if not delta:\n                    continue\n                payload = {"choices": [{"delta": {"content": delta}}]}\n                yield f"data: {json.dumps(payload)}\\n\\n"\n        except Exception as exc:  # noqa: BLE001\n            # The response has already begun, so the only way to report a\n            # mid-stream failure is inside the stream itself.\n            yield f"data: {json.dumps({\'error\': str(exc)})}\\n\\n"\n        yield "data: [DONE]\\n\\n"\n\n    return StreamingResponse(\n        sse(),\n        media_type="text/event-stream",\n        headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"},\n    )\n\n\n@app.post("/generate")\ndef generate_endpoint(req: GenerateRequest, _: str = Depends(check_token)) -> dict:\n    try:\n        text = get_engine().generate(\n            req.prompt,\n            temperature=req.temperature,\n            max_tokens=req.max_tokens,\n            thinking=req.thinking,\n            reasoning_effort=req.reasoning_effort,\n            response_format=req.response_format,\n        )\n    except Exception as exc:  # noqa: BLE001\n        raise HTTPException(502, f"Model error: {exc}") from exc\n    return {"content": text}\n\n\n# Plans in flight, by cache key.\n#\n# The cache was checked once, on the way in, and generate_plan() then queued on\n# the engine lock. So a second identical request arriving mid-generation missed\n# the cache, waited for the first to finish, and generated the whole plan again\n# — measured: a request cut by the gateway at 300.8s, retried 90s later, came\n# back cached=false with latency_sec 296.2. Two full generations for one plan.\n#\n# That matters because the gateway cuts at 300s and a plan takes about that\n# long, so "fire, get cut, ask again" is the natural way to use this endpoint.\n# It has to collect the first answer rather than start a second one.\n_inflight: Dict[str, threading.Event] = {}\n_inflight_results: Dict[str, Any] = {}\n_inflight_guard = threading.Lock()\n\n\ndef _release(key: str, plan: Optional[dict]) -> None:\n    """Wake everyone waiting on this key, whether it worked or not.\n\n    The result is held briefly as well as cached, because a waiter that arrives\n    after the event is set but before the cache write is visible would\n    otherwise see nothing and report a failure for a plan that succeeded.\n    """\n    with _inflight_guard:\n        event = _inflight.pop(key, None)\n        if plan is not None:\n            _inflight_results[key] = plan\n        if event is not None:\n            event.set()\n\n\n@app.post("/director")\ndef director_endpoint(req: DirectorRequest, _: str = Depends(check_token)) -> dict:\n    key = get_cache_key(req.script, req.style, MODEL_ID, SCHEMA_VERSION, f"{req.cues}|{req.brief}|{req.reasoning_effort}")\n    if not req.no_cache:\n        cached = get_cached_result(key)\n        if cached is not None:\n            return {"success": True, "cached": True, "plan": cached, **cached}\n\n        # Is this exact plan already being generated? Wait for it instead of\n        # asking the GPU for the same thing twice.\n        waiter = None\n        with _inflight_guard:\n            if key in _inflight:\n                waiter = _inflight[key]\n            else:\n                _inflight[key] = threading.Event()\n\n        if waiter is not None:\n            # No timeout: the caller\'s own client decides how long to wait, and\n            # the work is already running either way.\n            waiter.wait()\n            done = get_cached_result(key)\n            if done is None:\n                done = _inflight_results.get(key)\n            if done is not None:\n                return {"success": True, "cached": True, "joined": True, "plan": done, **done}\n            raise HTTPException(502, "Director: the request this one joined failed.")\n\n    try:\n        plan, latency = get_engine().generate_plan(\n            script=req.script,\n            style=req.style,\n            title=req.title,\n            cues=req.cues,\n            brief=req.brief,\n            max_tokens=req.max_tokens,\n            reasoning_effort=req.reasoning_effort,\n            plan_tokens=req.plan_tokens,\n        )\n    except Exception as exc:  # noqa: BLE001\n        # Release anyone waiting on this key before failing, or they wait for a\n        # result that is never coming. finally would be tidier but must not run\n        # before set_cached_result below.\n        _release(key, None)\n        raise HTTPException(502, f"Director failed: {exc}") from exc\n\n    set_cached_result(key, plan)\n    _release(key, plan)\n    # `plan` is also splatted at the top level so a caller can read `scenes`\n    # straight off the response without knowing about this envelope.\n    return {\n        "success": True,\n        "cached": False,\n        "latency_sec": round(latency, 2),\n        "plan": plan,\n        **plan,\n    }\n'
Path('director/api.py').write_text(SRC, encoding='utf-8')
print('  wrote director/api.py', len(SRC), 'bytes')

In [ ]:
# director/tests.py — generated from the package by build_notebook.py
SRC = '"""Contract tests. No GPU, no model — these run anywhere in about a second.\n\nThe thing worth testing here is not that pydantic works, it is that the schema\nstill lines up with what AETHER\'s parseVideoPlan() reads. That parser drops a\nscene with no numeric index and silently rewrites an unknown visualType, so a\ndrift between the two files produces an empty storyboard and no error.\n\n    python -m unittest director.tests -v\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport unittest\n\nfrom .cache import get_cache_key\nfrom .inference import strip_thinking\nfrom .schema import (\n    VideoPlan,\n    archive_query_advice,\n    get_json_schema,\n    plan_violations,\n    scene_violations,\n)\n\n# Mirrors the vocabularies in public/prompts.js. If a test here fails after you\n# edit that file, the fix is to change both, not to loosen the test.\nAETHER_VISUAL_TYPES = {\n    "stock_video", "stock_photo", "stock_text", "editorial_text",\n    "t2v", "broll", "presenter",\n    "stickman", "whiteboard", "chart", "map", "timeline", "diagram",\n}\nAETHER_SCENE_FIELDS = {\n    "index", "visualType", "stockRequirements", "textOverlay", "graphic",\n    "hostOverlay", "shotType", "cameraMovement", "motion", "emotion",\n    "transition", "note",\n    # Present only when the narration has been transcribed. Validated against\n    # the audio duration by public/transcription/timing.js, never trusted raw.\n    "timelineStart", "timelineEnd",\n}\n\nMINIMAL_PLAN = {\n    "strategy": "Footage for the concrete beats, a chart for the one number.",\n    "warnings": [],\n    "scenes": [\n        {\n            "index": 0,\n            "visualType": "stock_video",\n            "stockRequirements": {\n                "concept": "A shopper faced with higher prices.",\n                "queries": ["woman shopping for groceries", "supermarket price label close up"],\n                "fallbackQueries": ["grocery store aisle"],\n                "subjectCategory": "HUMAN",\n                "minimumDuration": 3.0,\n                # A present-day beat, so it names modern libraries. Required\n                # since 2.1: a stock beat with no source decision routes\n                # nowhere, and every library then gets the same generic search.\n                "sourceStrategy": "modern_stock",\n                "preferredSources": ["pexels", "pixabay"],\n            },\n            "shotType": "Medium",\n            "cameraMovement": "Slow Push In",\n            "motion": "She lifts an item and puts it back.",\n            "emotion": "resigned",\n            "transition": "cut",\n        },\n        {\n            "index": 1,\n            "visualType": "chart",\n            "graphic": {\n                "title": "Food prices",\n                "subtitle": "",\n                "items": ["2021: 61", "2024: 87"],\n            },\n            "transition": "dissolve",\n        },\n    ],\n}\n\n\nclass TestSchemaShape(unittest.TestCase):\n    def test_json_schema_is_object_with_scenes(self):\n        schema = get_json_schema()\n        self.assertEqual(schema["type"], "object")\n        self.assertIn("scenes", schema["properties"])\n\n    def test_objects_forbid_extra_keys(self):\n        """Guided decoding follows the grammar; an open object invites invention."""\n\n        def walk(node):\n            if isinstance(node, dict):\n                if node.get("type") == "object":\n                    self.assertIs(\n                        node.get("additionalProperties"), False,\n                        msg=f"object left open: {json.dumps(node)[:120]}",\n                    )\n                for value in node.values():\n                    walk(value)\n            elif isinstance(node, list):\n                for value in node:\n                    walk(value)\n\n        walk(get_json_schema())\n\n    def test_inlined_schema_has_no_refs(self):\n        """llama.cpp compiles the schema to a grammar and chokes on $ref."""\n        blob = json.dumps(get_json_schema(inline=True))\n        self.assertNotIn("$ref", blob)\n        self.assertNotIn("$defs", blob)\n\n    def test_inlined_schema_keeps_the_vocabulary(self):\n        """Inlining must not lose the enums that keep the model in bounds."""\n        scene = get_json_schema(inline=True)["properties"]["scenes"]["items"]\n        self.assertEqual(set(scene["properties"]["visualType"]["enum"]), AETHER_VISUAL_TYPES)\n        self.assertEqual(set(scene["properties"]), AETHER_SCENE_FIELDS)\n\n    def test_inlined_schema_still_forbids_extra_keys(self):\n        def walk(node):\n            if isinstance(node, dict):\n                if node.get("type") == "object":\n                    self.assertIs(node.get("additionalProperties"), False)\n                for value in node.values():\n                    walk(value)\n            elif isinstance(node, list):\n                for value in node:\n                    walk(value)\n\n        walk(get_json_schema(inline=True))\n\n    def test_scene_fields_match_aether_parser(self):\n        fields = set(VideoPlan.model_json_schema()["$defs"]["Scene"]["properties"])\n        self.assertEqual(\n            fields, AETHER_SCENE_FIELDS,\n            "Scene fields drifted from parseVideoPlan() in public/prompts.js",\n        )\n\n    def test_visual_types_match_aether_vocabulary(self):\n        enum = set(VideoPlan.model_json_schema()["$defs"]["Scene"]["properties"]["visualType"]["enum"])\n        self.assertEqual(\n            enum, AETHER_VISUAL_TYPES,\n            "visualType drifted from VISUAL_TYPES in public/prompts.js",\n        )\n\n\nclass TestValidation(unittest.TestCase):\n    def test_minimal_plan_validates(self):\n        plan = VideoPlan(**MINIMAL_PLAN)\n        self.assertEqual(len(plan.scenes), 2)\n        self.assertEqual(plan.scenes[0].stockRequirements.queries[0], "woman shopping for groceries")\n\n    def test_defaults_fill_in(self):\n        """AETHER reads every scene field, so none of them may be absent."""\n        scene = VideoPlan(**MINIMAL_PLAN).scenes[1]\n        self.assertEqual(scene.hostOverlay, "none")\n        self.assertEqual(scene.shotType, "Medium")\n        self.assertEqual(scene.cameraMovement, "Static")\n\n    def test_index_is_required(self):\n        """A scene with no index is dropped by AETHER, so reject it here."""\n        broken = json.loads(json.dumps(MINIMAL_PLAN))\n        del broken["scenes"][0]["index"]\n        with self.assertRaises(Exception):\n            VideoPlan(**broken)\n\n    def test_unknown_visual_type_rejected(self):\n        broken = json.loads(json.dumps(MINIMAL_PLAN))\n        broken["scenes"][0]["visualType"] = "ai_generated_clip"\n        with self.assertRaises(Exception):\n            VideoPlan(**broken)\n\n    def test_empty_scenes_rejected(self):\n        with self.assertRaises(Exception):\n            VideoPlan(strategy="", warnings=[], scenes=[])\n\n\nclass TestPayloadMatchesType(unittest.TestCase):\n    """The failure seen on Kaggle: a type chosen without the payload it draws from.\n\n    The grammar cannot express this rule — JSON Schema has no way to make one\n    field\'s presence depend on another field\'s value — so it has to be caught\n    after generation and either re-asked or repaired.\n    """\n\n    def _scene(self, **over):\n        base = {"index": 0, "visualType": "broll"}\n        base.update(over)\n        return base\n\n    def test_editorial_text_without_overlay_is_a_violation(self):\n        problems = scene_violations(self._scene(visualType="editorial_text"))\n        self.assertTrue(any("textOverlay.text" in p for p in problems), problems)\n\n    def test_stock_video_without_queries_is_a_violation(self):\n        problems = scene_violations(self._scene(visualType="stock_video"))\n        self.assertTrue(any("stockRequirements.queries" in p for p in problems), problems)\n\n    def test_chart_without_items_is_a_violation(self):\n        problems = scene_violations(self._scene(visualType="chart"))\n        self.assertTrue(any("graphic.items" in p for p in problems), problems)\n\n    def test_stock_text_needs_both(self):\n        # Names the two things it means rather than counting problems. The count\n        # was a proxy that broke the moment the contract grew a third rule, and\n        # a test that fails when an unrelated rule is added is a test that will\n        # be edited to make it pass rather than read.\n        problems = scene_violations(self._scene(visualType="stock_text"))\n        self.assertTrue(any("stockRequirements.queries" in p for p in problems), problems)\n        self.assertTrue(any("textOverlay.text" in p for p in problems), problems)\n\n    def test_stock_beat_without_a_source_decision_is_a_violation(self):\n        """Left optional, the model omits it and nothing routes the beat.\n\n        Measured against the live 27B: with these fields optional, every beat\n        came back with no preferred source at all — including one about the\n        Second World War — so the film archive was never reachable.\n        """\n        problems = scene_violations(self._scene(visualType="stock_video"))\n        self.assertTrue(any("preferredSources" in p for p in problems), problems)\n        self.assertTrue(any("sourceStrategy" in p for p in problems), problems)\n\n    def test_archive_beat_without_archive_queries_is_advice_not_a_violation(self):\n        """It renders fine, so it must not trigger a whole-plan regeneration.\n\n        A violation re-asks the model for the ENTIRE plan. Measured live, one\n        beat costs about 150s, so paying that twice for a beat that draws\n        correctly is the wrong trade — the producer gets a note instead.\n        """\n        scene = self._scene(\n            visualType="stock_video",\n            stockRequirements={\n                "concept": "wartime production",\n                "queries": ["factory workers"],\n                "sourceStrategy": "archival",\n                "preferredSources": ["archive_org"],\n                "archiveQueries": [],\n            },\n        )\n        self.assertEqual(scene_violations(scene), [])\n\n        notes = archive_query_advice({"scenes": [scene]})\n        self.assertTrue(any("archiveQueries" in msg for _, msg in notes), notes)\n\n    def test_archive_beat_with_archive_queries_draws_no_advice(self):\n        notes = archive_query_advice({"scenes": [self._scene(\n            visualType="stock_video",\n            stockRequirements={\n                "concept": "wartime production",\n                "queries": ["factory workers"],\n                "sourceStrategy": "archival",\n                "preferredSources": ["archive_org"],\n                "archiveQueries": ["1940s wartime factory newsreel"],\n            },\n        )]})\n        self.assertEqual(notes, [])\n\n    def test_archive_beat_with_archive_queries_is_clean(self):\n        problems = scene_violations(self._scene(\n            visualType="stock_video",\n            stockRequirements={\n                "concept": "wartime production",\n                "queries": ["factory workers"],\n                "sourceStrategy": "archival",\n                "preferredSources": ["archive_org"],\n                "archiveQueries": ["1940s wartime factory newsreel"],\n            },\n        ))\n        self.assertEqual(problems, [])\n\n    def test_modern_beat_needs_no_archive_queries(self):\n        problems = scene_violations(self._scene(\n            visualType="stock_video",\n            stockRequirements={\n                "concept": "shopping today",\n                "queries": ["supermarket checkout"],\n                "sourceStrategy": "modern_stock",\n                "preferredSources": ["pexels", "pixabay"],\n            },\n        ))\n        self.assertEqual(problems, [])\n\n    def test_types_needing_nothing_are_clean(self):\n        for visual in ("t2v", "broll", "presenter"):\n            self.assertEqual(scene_violations(self._scene(visualType=visual)), [])\n\n    def test_missing_index_is_a_violation(self):\n        scene = self._scene()\n        del scene["index"]\n        self.assertTrue(any("index" in p for p in scene_violations(scene)))\n\n    def test_a_good_plan_has_no_violations(self):\n        self.assertEqual(plan_violations(MINIMAL_PLAN), [])\n\n    def test_plan_violations_reports_the_scene_index(self):\n        broken = json.loads(json.dumps(MINIMAL_PLAN))\n        broken["scenes"][1]["graphic"] = None\n        found = plan_violations(broken)\n        self.assertEqual([i for i, _ in found], [1])\n\n    def test_model_rejects_a_mismatched_scene(self):\n        """Pydantic must refuse it too, or a bad plan reaches AETHER anyway."""\n        broken = json.loads(json.dumps(MINIMAL_PLAN))\n        broken["scenes"][0]["stockRequirements"] = None\n        with self.assertRaises(Exception):\n            VideoPlan(**broken)\n\n\nclass TestRepair(unittest.TestCase):\n    """The fallback when the model will not fix its own work."""\n\n    def setUp(self):\n        from .inference import DirectorInference\n\n        self.repair = DirectorInference._repair\n\n    def test_caption_is_taken_from_the_narration(self):\n        plan = {"scenes": [{"index": 0, "visualType": "editorial_text"}]}\n        self.repair(plan, [{"index": 0, "text": "Inflation quietly reduces what your paycheck can buy."}])\n        self.assertEqual(plan["scenes"][0]["visualType"], "editorial_text")\n        self.assertIn("Inflation", plan["scenes"][0]["textOverlay"]["text"])\n        self.assertEqual(plan_violations(plan), [])\n\n    def test_caption_is_capped_at_twelve_words(self):\n        plan = {"scenes": [{"index": 0, "visualType": "editorial_text"}]}\n        self.repair(plan, [{"index": 0, "text": " ".join(f"w{i}" for i in range(40))}])\n        self.assertEqual(len(plan["scenes"][0]["textOverlay"]["text"].split()), 12)\n\n    def test_stock_without_queries_is_demoted(self):\n        plan = {"scenes": [{"index": 0, "visualType": "stock_video"}]}\n        self.repair(plan, [{"index": 0, "text": "some narration"}])\n        self.assertEqual(plan["scenes"][0]["visualType"], "broll")\n        self.assertEqual(plan_violations(plan), [])\n\n    def test_empty_chart_is_demoted_rather_than_drawn_blank(self):\n        plan = {"scenes": [{"index": 0, "visualType": "chart"}]}\n        self.repair(plan, [{"index": 0, "text": "no numbers here"}])\n        self.assertEqual(plan["scenes"][0]["visualType"], "broll")\n\n    def test_missing_index_is_supplied(self):\n        plan = {"scenes": [{"visualType": "broll"}]}\n        self.repair(plan, [])\n        self.assertEqual(plan["scenes"][0]["index"], 0)\n        self.assertEqual(plan_violations(plan), [])\n\n    def test_every_repair_is_recorded_as_a_warning(self):\n        plan = {"scenes": [{"index": 0, "visualType": "chart"}]}\n        notes = self.repair(plan, [])\n        self.assertTrue(notes)\n        self.assertEqual(plan["warnings"], notes)\n\n    def test_repaired_plan_validates(self):\n        plan = json.loads(json.dumps(MINIMAL_PLAN))\n        plan["scenes"][0]["stockRequirements"] = None\n        plan["scenes"][1]["graphic"] = None\n        self.repair(plan, [{"index": 0, "text": "a"}, {"index": 1, "text": "b"}])\n        VideoPlan(**plan)\n\n\nclass TestStripThinking(unittest.TestCase):\n    """Recovering the answer from a reply that begins mid-thought.\n\n    This model\'s chat template ends the prompt with a bare `<think>\\\\n`, so\n    generation starts already inside the block and only the CLOSING tag is ever\n    generated. A regex looking for a matched pair matches nothing, and /generate\n    returned several hundred tokens of deliberation where a title was expected.\n    """\n\n    def test_closing_tag_with_no_opener(self):\n        raw = "1. Analyse the request\\n2. Draft options\\n</think>\\n\\nWhy Your Money Buys Less"\n        self.assertEqual(strip_thinking(raw), "Why Your Money Buys Less")\n\n    def test_matched_pair_still_works(self):\n        self.assertEqual(strip_thinking("<think>hmm</think>\\n\\nAnswer"), "Answer")\n\n    def test_last_closing_tag_wins(self):\n        raw = "think a</think>mid</think>\\n\\nFinal"\n        self.assertEqual(strip_thinking(raw), "Final")\n\n    def test_plain_text_is_untouched(self):\n        self.assertEqual(strip_thinking("Just an answer."), "Just an answer.")\n\n    def test_truncated_thought_yields_nothing(self):\n        """No closing tag means generation stopped mid-thought — there is no answer."""\n        self.assertEqual(strip_thinking("<think>still reasoning and then it stopped"), "")\n\n    def test_answer_containing_the_word_think_survives(self):\n        raw = "</think>\\n\\nI think inflation is the answer."\n        self.assertEqual(strip_thinking(raw), "I think inflation is the answer.")\n\n    def test_empty_and_none(self):\n        self.assertEqual(strip_thinking(""), "")\n        self.assertEqual(strip_thinking(None), "")\n\n    def test_json_after_a_thought_is_preserved(self):\n        raw = \'</think>\\n\\n{"scenes": []}\'\n        self.assertEqual(strip_thinking(raw), \'{"scenes": []}\')\n\n\nclass TestCache(unittest.TestCase):\n    def test_key_is_stable(self):\n        args = ("script", "finance", "model-1", "2.0", "")\n        self.assertEqual(get_cache_key(*args), get_cache_key(*args))\n\n    def test_schema_version_changes_key(self):\n        a = get_cache_key("s", "finance", "m", "1.0", "")\n        b = get_cache_key("s", "finance", "m", "2.0", "")\n        self.assertNotEqual(a, b, "a schema change must miss the cache, not serve a stale shape")\n\n\nif __name__ == "__main__":\n    unittest.main()\n\n\nclass TestArchivalContract(unittest.TestCase):\n    """Archival beats must arrive usable, enforced by the grammar not by retry.\n\n    The Director is the only thing that can decide why an archive earns this\n    beat, what to search it for, and which moment inside a whole film to use.\n    Left optional, a constrained model omits all three — measured. A validator\n    would catch it, but a violation re-asks for the ENTIRE plan at ~150s a beat,\n    so the schema makes the omission unspellable instead.\n    """\n\n    ARCHIVAL = {\n        "concept": "American factories converting to war production",\n        "queries": ["factory assembly line workers"],\n        "sourceStrategy": "archival",\n        "preferredSources": ["archive_org"],\n        "archiveQueries": ["1940s wartime factory newsreel"],\n        "excerpt": {"required": True, "targetDuration": 6.0,\n                    "selectionIntent": "workers operating wartime machinery"},\n        "editorialPurpose": "historical context for the production figures",\n        "timePeriod": {"from_year": 1941, "to_year": 1945, "label": "Second World War"},\n    }\n    MODERN = {\n        "concept": "Shopping today",\n        "queries": ["supermarket checkout"],\n        "sourceStrategy": "modern_stock",\n        "preferredSources": ["pexels", "pixabay"],\n    }\n\n    def _plan(self, requirements):\n        plan = json.loads(json.dumps(MINIMAL_PLAN))\n        plan["scenes"][0]["stockRequirements"] = requirements\n        return plan\n\n    def test_a_complete_archival_beat_validates(self):\n        scene = VideoPlan(**self._plan(self.ARCHIVAL)).scenes[0]\n        self.assertEqual(scene.stockRequirements.sourceStrategy, "archival")\n        self.assertEqual(scene.stockRequirements.excerpt.selectionIntent,\n                         "workers operating wartime machinery")\n\n    def test_a_modern_beat_needs_none_of_it(self):\n        scene = VideoPlan(**self._plan(self.MODERN)).scenes[0]\n        self.assertEqual(scene.stockRequirements.sourceStrategy, "modern_stock")\n        self.assertFalse(hasattr(scene.stockRequirements, "excerpt"))\n\n    def test_archival_without_an_excerpt_is_rejected(self):\n        broken = dict(self.ARCHIVAL)\n        del broken["excerpt"]\n        with self.assertRaises(Exception):\n            VideoPlan(**self._plan(broken))\n\n    def test_archival_without_archive_queries_is_rejected(self):\n        broken = dict(self.ARCHIVAL)\n        broken["archiveQueries"] = []\n        with self.assertRaises(Exception):\n            VideoPlan(**self._plan(broken))\n\n    def test_archival_without_an_editorial_purpose_is_rejected(self):\n        broken = dict(self.ARCHIVAL)\n        broken["editorialPurpose"] = ""\n        with self.assertRaises(Exception):\n            VideoPlan(**self._plan(broken))\n\n    def test_time_period_stays_optional(self):\n        """A guessed decade searches the wrong decade. Absent beats invented."""\n        without = dict(self.ARCHIVAL)\n        del without["timePeriod"]\n        scene = VideoPlan(**self._plan(without)).scenes[0]\n        self.assertIsNone(scene.stockRequirements.timePeriod)\n\n    def test_archive_org_is_repaired_into_the_sources(self):\n        """The beat is already archival; omitting the library is a slip, not a\n        different decision — and rejecting it would cost a whole re-plan."""\n        slipped = dict(self.ARCHIVAL, preferredSources=["pexels"])\n        scene = VideoPlan(**self._plan(slipped)).scenes[0]\n        self.assertIn("archive_org", scene.stockRequirements.preferredSources)\n\n    def test_the_grammar_carries_the_rule_not_just_the_validator(self):\n        """If this only held in pydantic, the model could still omit the fields\n        and we would pay a re-plan to find out."""\n        req = (get_json_schema(inline=True)["properties"]["scenes"]["items"]\n               ["properties"]["stockRequirements"])\n        branches = [b for alt in req["anyOf"] for b in alt.get("oneOf", [])]\n        archival = next(b for b in branches\n                        if b["properties"]["sourceStrategy"].get("const") == "archival")\n        for field in ("archiveQueries", "excerpt", "editorialPurpose"):\n            self.assertIn(field, archival["required"], f"{field} not forced by the grammar")\n\n        modern = next(b for b in branches\n                      if "modern_stock" in (b["properties"]["sourceStrategy"].get("enum") or []))\n        for field in ("archiveQueries", "excerpt", "editorialPurpose"):\n            self.assertNotIn(field, modern["properties"],\n                             f"{field} offered to a modern beat, which would teach it to invent one")\n\nclass TestReasoningNeverReachesTheAnswer(unittest.TestCase):\n    """The leak that put a model thinking aloud into a finished script.\n\n    A generated script came back beginning "We need answer user\'s request. Need\n    produce final only spoken narration, no headings etc." and continued through\n    a draft and a word-by-word count before reaching the narration. All of that\n    is the reasoning block, and none of it should ever leave the server.\n\n    The cause was that neither stripper knew whether the PROMPT had opened the\n    block. With reasoning on, the chat template ends on a bare <think>, so\n    generation starts inside it and only the CLOSING tag is ever produced. Text\n    with no tags at all is therefore not an answer — it is deliberation that\n    never closed.\n    """\n\n    LEAKED = (\n        "We need answer user\'s request. Need produce final only spoken "\n        "narration, no headings etc. Let\'s draft around 400. Count.\\n\\n"\n        "Imagine a single morning that turns a continent into a battlefield."\n    )\n\n    def test_untagged_output_is_not_an_answer_when_the_prompt_opened_the_block(self):\n        self.assertEqual(strip_thinking(self.LEAKED, started_inside=True), "")\n\n    def test_untagged_output_is_kept_when_reasoning_was_off(self):\n        # With thinking disabled the template pre-closes the block, so the model\n        # starts outside it and its output is the answer.\n        self.assertEqual(\n            strip_thinking("Imagine a single morning.", started_inside=False),\n            "Imagine a single morning.",\n        )\n\n    def test_a_closed_block_still_yields_what_follows(self):\n        self.assertEqual(\n            strip_thinking("thinking hard</think>The narration.", started_inside=True),\n            "The narration.",\n        )\n\n    def test_the_default_is_unchanged_for_existing_callers(self):\n        self.assertEqual(strip_thinking("plain answer"), "plain answer")\n\n\nclass TestStreamingReasoningGate(unittest.TestCase):\n    """The streaming half of the same bug.\n\n    _stream initialised in_thought=False directly beneath a comment saying it\n    must assume the opposite, so every reasoning token was streamed to the\n    caller until a closing tag arrived — and if none ever did, the entire\n    deliberation was delivered as the answer. Script generation streams, which\n    is why it leaked there.\n\n    Exercised through the real gate logic rather than a copy of it: the loop is\n    a closure over deltas, so this re-implements the caller, not the rule.\n    """\n\n    def _gate(self, deltas, thinking):\n        """Run the same gate _stream applies, over a list of deltas."""\n        in_thought = bool(thinking)\n        seen_close = False\n        out = []\n        for delta in deltas:\n            if not delta:\n                continue\n            if not seen_close and "</think>" in delta:\n                seen_close = True\n                in_thought = False\n                delta = delta.split("</think>", 1)[-1]\n                if not delta:\n                    continue\n            if "<think>" in delta:\n                in_thought = True\n                delta = delta.split("<think>")[0]\n            if in_thought:\n                if "</think>" not in delta:\n                    continue\n                in_thought = False\n                delta = delta.split("</think>")[-1]\n            if delta:\n                out.append(delta)\n        return "".join(out)\n\n    def test_reasoning_without_a_closing_tag_yields_nothing(self):\n        deltas = ["We need ", "answer user\'s ", "request. ", "Let\'s draft."]\n        self.assertEqual(self._gate(deltas, thinking=True), "")\n\n    def test_the_answer_after_the_closing_tag_is_yielded(self):\n        deltas = ["We need answer ", "user\'s request.", "</think>", "Imagine ", "a morning."]\n        self.assertEqual(self._gate(deltas, thinking=True), "Imagine a morning.")\n\n    def test_a_closing_tag_split_across_deltas_is_still_the_boundary(self):\n        # The tag can arrive as its own delta, which is the common case.\n        deltas = ["deliberating", "</think>", "The narration begins."]\n        self.assertEqual(self._gate(deltas, thinking=True), "The narration begins.")\n\n    def test_with_reasoning_off_everything_is_the_answer(self):\n        deltas = ["Imagine ", "a single ", "morning."]\n        self.assertEqual(self._gate(deltas, thinking=False), "Imagine a single morning.")\n\n    def test_the_old_behaviour_would_have_leaked(self):\n        """Guards the fix itself: starting outside the block reproduces the bug."""\n        deltas = ["We need ", "answer user\'s request."]\n        leaked = self._gate(deltas, thinking=False)\n        self.assertIn("We need", leaked)\n        self.assertEqual(self._gate(deltas, thinking=True), "")\n'
Path('director/tests.py').write_text(SRC, encoding='utf-8')
print('  wrote director/tests.py', len(SRC), 'bytes')

## Stage 5 — Compile and run the contract tests

No GPU needed. This is what catches the schema drifting away from AETHER's parser.

In [ ]:
import glob, py_compile, subprocess, sys

for path in sorted(glob.glob('director/*.py')):
    py_compile.compile(path, doraise=True)
    print('  compiles:', path)

result = subprocess.run([sys.executable, '-m', 'unittest', 'director.tests', '-v'],
                        capture_output=True, text=True)
print(result.stdout[-3000:])
print(result.stderr[-3000:])
if result.returncode != 0:
    raise RuntimeError('Contract tests failed — the schema no longer matches AETHER.')
print('Stage 5 PASSED')

## Stage 5b — Prove llama.cpp can reach the GPU

Fail here, not after a 20 GB download. This is the cell that catches the
`libcudart.so.12` import error and, more quietly, a CPU-only wheel — which
installs and imports perfectly and then runs the model at about one token per
second with no warning that anything is wrong.

In [ ]:
import sys

sys.path.insert(0, '.')
from director.inference import preload_cuda_libraries   # noqa: E402

loaded = preload_cuda_libraries()
print('preloaded CUDA libraries:')
for lib in loaded:
    print('   ', lib)
if not loaded:
    print('    (none needed — already on the loader path)')

try:
    import llama_cpp
except Exception as exc:
    raise RuntimeError(
        f'llama_cpp will not import: {exc}\n'
        'The CUDA runtime is missing or the wheel does not match it. '
        'Re-run Stage 2, and check the wheel index matches this image\'s CUDA.'
    ) from exc

print(f'\nllama_cpp {llama_cpp.__version__}')

# supports_gpu_offload() is the honest test. A CPU-only build imports fine and
# silently ignores n_gpu_layers, so without this you find out from the token
# rate an hour later.
gpu_ok = False
try:
    gpu_ok = bool(llama_cpp.llama_supports_gpu_offload())
except Exception as exc:
    print('could not query GPU offload support:', exc)

print('GPU offload supported:', gpu_ok)
if not gpu_ok:
    raise RuntimeError(
        'This llama-cpp-python build cannot offload to the GPU — it is a '
        'CPU-only wheel. A 27B model will be unusably slow. Reinstall from '
        'the cu124 index in Stage 2.'
    )
print('\nStage 5b PASSED')

## A note on narration timing

There is deliberately no Whisper stage here.

The Director benefits enormously from knowing when each word was actually
spoken — a statistic card should land as the number is said, not at the top of
the beat that mentions it. But this is the wrong machine to find that out on.

Fish Speech generates the narration, and the Fish backend already keeps
faster_whisper resident for voice cloning. AETHER asks it for forced alignment
over `/v1/align` (see public/sync-engine.js), gets real word timings back from
the machine that made the audio, and sends them to `/director` as part of the
request. No second Whisper to deploy, no upload round trip, and nothing
competing with these 20 GB of weights for VRAM.

If the alignment endpoint is unavailable, AETHER falls back to per-chunk
durations with words distributed by syllable weight — and labels the timing
`estimated` rather than passing a guess off as measurement.

## Stage 6 — Download the GGUF (~20 GB, once per session)

In [ ]:
import os, re, shutil, threading, time
from huggingface_hub import hf_hub_download

# Decimal GB (10^9), because that is how HuggingFace lists these files.
# Anything comparing against it must divide by 1e9, NOT by 1024**3 — the two
# differ by 7%, which is enough to fail a complete download: Q5_K_M is 19.8 GB
# and 18.4 GiB, and a guard that mixed the units rejected the finished file.
# Ask the repo what it actually holds, rather than hardcoding a filename and a
# size. Both go stale: unsloth re-quantized this model and every plain K-quant
# became a UD ("Unsloth Dynamic") build, so
#   Qwen3.8-27B-Q5_K_M.gguf   ->  404
#   Qwen3.8-27B-UD-Q5_K_M.gguf -> the real file
# A hardcoded name cannot survive that, and a hardcoded size is a second thing
# to get wrong — the sizes here came from the repo listing and were already
# once compared in the wrong unit.
from huggingface_hub import HfApi

_api = HfApi()
_files = [f for f in _api.list_repo_files(MODEL_REPO) if f.endswith('.gguf')]

def _pick(quant):
    # The file for this quant, preferring an exact name and then a UD build.
    #
    # Ordered deliberately: an exact match wins, then the dynamic build of
    # the same quant, and only then anything containing the name - so asking
    # for Q5_K_M can never quietly return Q5_K_S because it sorted first.
    base = MODEL_REPO.split('/')[-1].replace('-GGUF', '')
    for candidate in (f'{base}-{quant}.gguf', f'{base}-UD-{quant}.gguf'):
        if candidate in _files:
            return candidate
    loose = [f for f in _files if quant in f and '/' not in f]
    return loose[0] if loose else None

MODEL_FILE = _pick(QUANT)
if not MODEL_FILE:
    _offered = sorted({re.sub(r'^.*?-(UD-)?(Q\d[^.]*|BF16|F16)\.gguf$', r'\2', f)
                       for f in _files if '/' not in f and 'mmproj' not in f})
    raise RuntimeError(
        f'{MODEL_REPO} has no file for {QUANT}. It offers: {", ".join(_offered)}.\n'
        f'Set QUANT in Stage 3 to one of those.'
    )

# The size comes from the same listing, so it cannot disagree with the file.
_info = _api.model_info(MODEL_REPO, files_metadata=True)
_entry = next((x for x in _info.siblings if x.rfilename == MODEL_FILE), None)
EXPECTED_BYTES = getattr(_entry, 'size', None) or 0
EXPECTED_GIB = EXPECTED_BYTES / 1024**3 if EXPECTED_BYTES else 0
EXPECTED_GB  = EXPECTED_BYTES / 1e9 if EXPECTED_BYTES else 0
print(f'Resolved {QUANT} -> {MODEL_FILE}'
      + (f'  ({EXPECTED_GIB:.1f} GiB / {EXPECTED_GB:.1f} GB)' if EXPECTED_BYTES else ''))
if not EXPECTED_BYTES:
    # Without a published size there is nothing to verify against; say so
    # rather than inventing a threshold.
    print('  [WARN] the repo did not publish a size, so the completeness check '
          'will only verify the GGUF magic bytes.')

os.makedirs(MODEL_DIR, exist_ok=True)
free_gib = shutil.disk_usage(MODEL_DIR).free / 1024**3
print(f'Free on {MODEL_DIR}: {free_gib:.1f} GiB   |   need ~{EXPECTED_GIB:.1f} GiB '
      f'({EXPECTED_GB:.1f} GB as HuggingFace lists it)')
if free_gib < EXPECTED_GIB * 1.15:
    raise RuntimeError(
        f'Not enough disk: {free_gib:.1f} GiB free, {QUANT} needs about '
        f'{EXPECTED_GIB:.1f} GiB. Drop to a smaller quant in Stage 3.'
    )

# Downloading straight into local_dir avoids the older two-step behaviour that
# also filled the HF cache — which is the same 20 GB again, on a disk that
# does not have it.
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '0')

t0 = time.time()
done = threading.Event()
target = os.path.join(MODEL_DIR, MODEL_FILE)

def _beat():
    # hf_hub_download's progress bar does not always survive Kaggle's output
    # handling, and twenty silent minutes looks exactly like a dead kernel.
    while not done.wait(30):
        got = 0
        for root, _dirs, files in os.walk(MODEL_DIR):
            for f in files:
                try:
                    got += os.path.getsize(os.path.join(root, f))
                except OSError:
                    pass
        got_gib = got / 1024**3
        print(f'  [download] {got_gib:5.1f} / ~{EXPECTED_GIB:.1f} GiB   '
              f'({int(time.time()-t0)}s elapsed)', flush=True)

threading.Thread(target=_beat, daemon=True).start()
try:
    MODEL_PATH = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, local_dir=MODEL_DIR)
finally:
    done.set()

# Verify what actually landed, rather than only reporting it.
#
# A short or interrupted download passed this stage and then failed in
# Stage 7 as "ValueError: Failed to load model from file", which points at
# the loader and says nothing about the 20 GB that never arrived. Checked
# here, the message names the real problem and the fix.
got_bytes = os.path.getsize(MODEL_PATH)
got_gib = got_bytes / 1024**3
with open(MODEL_PATH, 'rb') as fh:
    magic = fh.read(4)

print(f'\nPath : {MODEL_PATH}')
print(f'Size : {got_gib:.1f} GiB of ~{EXPECTED_GIB:.1f} GiB expected '
      f'({got_bytes/1e9:.1f} GB of ~{EXPECTED_GB:.1f} GB)   ({int(time.time()-t0)}s)')
print(f'Magic: {magic!r}')

if magic != b'GGUF':
    raise RuntimeError(
        f'{MODEL_PATH} does not begin with the GGUF magic bytes (got {magic!r}). '
        'That file is not a usable model. Delete it and run this stage again.'
    )
# 4% covers the gap between the quant table above and the real file; further
# off than that is a truncated transfer, not a rounding difference.
if EXPECTED_GIB and got_gib < EXPECTED_GIB * 0.96:
    raise RuntimeError(
        f'{MODEL_PATH} is {got_gib:.1f} GiB but {MODEL_FILE} should be about '
        f'{EXPECTED_GIB:.1f} GiB - the download did not finish. Remove it and '
        f'run this stage again:  rm -rf {MODEL_DIR}'
    )

os.environ['DIRECTOR_MODEL_PATH'] = MODEL_PATH
print('\nStage 6 PASSED')

## Stage 7 — Load the model and smoke test

The CUDA runtime is preloaded first; without it `import llama_cpp` fails with `libcudart.so.12: cannot open shared object file`.

In [ ]:
import os
import time
from director.inference import DirectorInference, preload_cuda_libraries, quiet_backend_logs

# Check the file before the loader does. llama.cpp reports every reason for
# failing to load as the same ValueError, so an absent or truncated GGUF is
# indistinguishable from an incompatible one unless it is checked here.
if not MODEL_PATH or not os.path.exists(MODEL_PATH):
    raise RuntimeError(
        f'No model at {MODEL_PATH!r}. Kaggle clears /tmp between sessions, so '
        'Stage 6 has to run in this session before this one.'
    )
_gb = os.path.getsize(MODEL_PATH) / 1024**3
with open(MODEL_PATH, 'rb') as _fh:
    _magic = _fh.read(4)
print(f'model: {MODEL_PATH}  ({_gb:.1f} GB, magic {_magic!r})')
if _magic != b'GGUF':
    raise RuntimeError(
        f'{MODEL_PATH} is not a GGUF file (starts with {_magic!r}). Delete it '
        'and re-run Stage 6.'
    )
if _gb < 5:
    raise RuntimeError(
        f'{MODEL_PATH} is only {_gb:.1f} GB, far short of any supported quant - '
        'the download was interrupted. Delete it and re-run Stage 6.'
    )

loaded = preload_cuda_libraries()
print('preloaded CUDA libs:')
for lib in loaded:
    print('  ', lib)
if not loaded:
    print('   (none needed — already on the loader path)')

import threading
print('\nLoading 20 GB across the GPUs — 2-5 min, and llama.cpp logs as it goes.', flush=True)
t0 = time.time()
_done = threading.Event()
def _beat():
    while not _done.wait(30):
        print(f'  [loading] {int(time.time()-t0)}s elapsed...', flush=True)
threading.Thread(target=_beat, daemon=True).start()
try:
    engine = DirectorInference(model_path=MODEL_PATH, n_ctx=N_CTX, n_gpu_layers=-1, verbose=True)
finally:
    _done.set()
print(f'\nLoaded in {time.time()-t0:.0f}s')

# Loading is done, so the per-token chatter has stopped being progress and
# started being noise. Left on, a single plan prints thousands of "CUDA Graph
# id N reused" lines, which is what pushes the notebook output past the size
# where Kaggle starts refusing to save the draft.
print('backend logging:', 'quiet' if quiet_backend_logs() else 'still verbose (log hook unavailable)')

import torch
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f'  GPU {i}: {(total-free)/1024**3:.1f}/{total/1024**3:.1f} GB used')

# Liveness, not reasoning. thinking=False matters here: with reasoning on, this
# model opens a <think> block and spends its whole budget in it, so asking for a
# three-word answer in 64 tokens produced 64 tokens of deliberation and no
# answer at all. That used to be returned AS the answer — the stage printed
# "Response: 'Thinking:\n\n1. **Analyze the Request:**...'" and passed — and now
# it correctly raises instead. Either way the request was wrong: a smoke test
# should ask the cheapest question that proves the engine decodes.
t0 = time.time()
text = engine.chat(
    [{'role': 'user', 'content': 'Say exactly: INFERENCE OK'}],
    max_tokens=32,
    thinking=False,
)
print(f'\nResponse: {text!r}')
print(f'Latency : {time.time()-t0:.1f}s  (no reasoning — this is a liveness check)')
if not text.strip():
    raise RuntimeError('Empty response — the model loaded but generated nothing.')

# One reasoning call too, with room to finish, because thinking=False exercises
# a different path through the chat template and Stage 8 depends on the other.
t1 = time.time()
thought = engine.chat(
    [{'role': 'user', 'content': 'In one sentence: why is 1939 the year the war began in Europe?'}],
    max_tokens=1200,
    reasoning_effort='low',
)
print(f'\nWith reasoning: {thought.strip()[:160]!r}')
print(f'Latency : {time.time()-t1:.1f}s')
if not thought.strip():
    raise RuntimeError(
        'Reasoning produced no answer. The model spent its budget deliberating '
        'and never closed the block — raise max_tokens.'
    )
print('\nStage 7 PASSED')

## Stage 7b — Generation speed

Worth knowing before you wait on a storyboard: everything downstream scales off this number.

In [ ]:
import time

t0 = time.time()
first = None
count = 0
for delta in engine.chat(
    [{'role': 'user', 'content': 'Explain what inflation is, in about 150 words.'}],
    max_tokens=256, stream=True,
):
    if first is None:
        first = time.time()
    print(delta, end='', flush=True)
    count += 1

if first is None:
    raise RuntimeError('Stream produced nothing.')
gen = time.time() - first
print(f'\n\nTTFT   : {first-t0:.2f}s')
print(f'Chunks : {count}')
print(f'Speed  : {count/gen:.1f} chunks/s over {gen:.1f}s')
print('\nA storyboard is a few thousand tokens, so budget accordingly.')

## Stage 8 — Structured plan against the AETHER grammar

In [ ]:
import json
from director.schema import VideoPlan

# `engine` is the one loaded in Stage 7. Constructing a second
# DirectorInference would load another ~20 GB copy of the weights, which is
# more VRAM than the machine has.

TEST_SCRIPT = (
    'Inflation quietly reduces what your paycheck can buy over time. '
    'As prices rise, the same salary buys fewer groceries and less fuel. '
    'Food prices rose from an index of 61 in 2021 to 87 in 2024. '
    'Central banks respond by raising interest rates, which affects mortgages and spending.'
)

# A grammar-constrained plan is thousands of tokens at roughly ten a second, and
# with the backend quiet there is nothing on screen while it works. Without this
# the stage is indistinguishable from a hang — which is exactly how it was read.
import threading
import time as _time

_planning = threading.Event()
_t0 = _time.time()
def _beat():
    while not _planning.wait(20):
        print(f'  [planning] {int(_time.time()-_t0)}s elapsed...', flush=True)
threading.Thread(target=_beat, daemon=True).start()
print('Planning — expect a few minutes; the source fields are required now, '
      'so there is more to generate than before.', flush=True)
try:
    plan, latency = engine.generate_plan(script=TEST_SCRIPT, style='finance', title='Inflation explained')
finally:
    _planning.set()

validated = VideoPlan(**plan)

print(f'Latency : {latency:.1f}s')
print(f'Scenes  : {len(validated.scenes)}')
print(f'Strategy: {validated.strategy}\n')
for s in validated.scenes:
    detail = ''
    if s.stockRequirements:
        detail = ' | '.join(s.stockRequirements.queries[:2])
    elif s.graphic:
        detail = ', '.join(s.graphic.items[:3])
    elif s.textOverlay:
        detail = s.textOverlay.text
    print(f'  [{s.index}] {s.visualType:15} {detail}')
    # The source decision, printed because it is the thing that was silently
    # missing: with these fields optional the model omitted them on every beat
    # and nothing was ever routed to the film archive.
    r = s.stockRequirements
    if r:
        print(f'       source: {r.sourceStrategy} -> {", ".join(r.preferredSources) or "(none)"}')
        # getattr, because stockRequirements is a union discriminated on
        # sourceStrategy: a modern beat has no archiveQueries, excerpt or
        # editorialPurpose at all — that is the point of the split, not a gap
        # to fill in. Reading them directly raises AttributeError on exactly
        # the beats that are behaving correctly.
        archive_queries = getattr(r, 'archiveQueries', None)
        if archive_queries:
            print(f'       archive: {" | ".join(archive_queries[:2])}')
        excerpt = getattr(r, 'excerpt', None)
        if excerpt:
            print(f'       excerpt: {excerpt.targetDuration:.0f}s — {excerpt.selectionIntent}')
        purpose = getattr(r, 'editorialPurpose', '')
        if purpose:
            print(f'       why    : {purpose}')
        period = getattr(r, 'timePeriod', None)
        if period:
            print(f'       period : {period.label or ""} '
                  f'{period.from_year or ""}-{period.to_year or ""}')

director_plan = plan
print('\nStage 8 PASSED')

## Stage 9 — AETHER compatibility

Re-implements what `parseVideoPlan()` does, so a plan that would arrive empty in the app fails here instead.

In [ ]:
from director.schema import plan_violations

# Uses the same check the director itself runs, rather than a second copy that
# can drift from it. If this fails, the retry-and-repair path inside
# generate_plan did not save the plan — which is worth knowing loudly.
problems = plan_violations(director_plan)
scenes = director_plan.get('scenes', [])

print(f'{len(scenes)} scenes, '
      f'{len(set(s.get("visualType") for s in scenes))} distinct visual types')
for s in scenes:
    detail = ''
    if (s.get('stockRequirements') or {}).get('queries'):
        detail = ' | '.join(s['stockRequirements']['queries'][:2])
    elif (s.get('graphic') or {}).get('items'):
        detail = ', '.join(s['graphic']['items'][:3])
    elif (s.get('textOverlay') or {}).get('text'):
        detail = s['textOverlay']['text']
    print(f'  [{s.get("index")}] {s.get("visualType"):15} {detail[:60]}')

for w in director_plan.get('warnings', []):
    print('  note:', w)

if problems:
    print()
    for i, msg in problems:
        print(f'  FAIL: scene {i}: {msg}')
    raise RuntimeError(f'{len(problems)} compatibility issue(s) — AETHER could not use this plan.')
print('\nStage 9 PASSED — plan is AETHER-compatible')

## Stage 10 — FastAPI + ngrok

The server runs **inside this notebook**, in a background thread, reusing the
model already in VRAM. Launching uvicorn as a subprocess the way a vLLM setup
would is not an option here: llama.cpp holds the model in the process that
loaded it, so a second process means a second 20 GB copy and an immediate OOM.

Leave this cell's kernel running for as long as you want AETHER to reach the
tunnel.

In [ ]:
import threading, time
import requests
import uvicorn
import director.api as api

# Hand the API the model that is already loaded, and the settings Stage 3 chose.
api._engine   = engine
api.MODEL_PATH = MODEL_PATH
api.N_CTX      = N_CTX
api.API_KEY    = DIRECTOR_API_KEY
api.MODEL_ID   = f'{MODEL_REPO}:{QUANT}'

server = uvicorn.Server(uvicorn.Config(
    api.app, host='0.0.0.0', port=API_PORT, log_level='warning',
    # A storyboard can hold a connection for minutes; the default keep-alive
    # would drop it mid-generation.
    timeout_keep_alive=1800,
))
threading.Thread(target=server.run, daemon=True).start()

health = None
for _ in range(30):
    time.sleep(2)
    try:
        health = requests.get(f'http://localhost:{API_PORT}/health', timeout=5).json()
        break
    except Exception:
        pass
if health is None:
    raise RuntimeError('FastAPI never became healthy.')
print('health:', health)
if not health.get('loaded'):
    raise RuntimeError('API is up but has no model — engine handoff failed.')

AUTH = {'Authorization': f'Bearer {DIRECTOR_API_KEY}'}

# Auth must actually be enforced, and /health must actually be open.
assert requests.post(f'http://localhost:{API_PORT}/generate',
                     json={'prompt': 'hi'}, timeout=30).status_code in (401, 403), 'auth not enforced!'

chat = requests.post(f'http://localhost:{API_PORT}/chat', headers=AUTH,
                     json={'messages': [{'role': 'user', 'content': 'Reply with one word: ready'}],
                           'max_tokens': 512}, timeout=600)
print('/chat     ', chat.status_code, chat.json()['choices'][0]['message']['content'][:60] if chat.ok else chat.text[:200])

gen = requests.post(f'http://localhost:{API_PORT}/generate', headers=AUTH,
                    json={'prompt': 'Give one YouTube title about inflation.', 'max_tokens': 1024}, timeout=600)
print('/generate ', gen.status_code, gen.json()['content'][:60] if gen.ok else gen.text[:200])

# The slowest probe by a wide margin, and the tunnel is not created until it
# returns — so a silent wait here reads as a dead stage right at the point where
# everything else has already worked. Stages 7 and 8 tick; this one did not.
import threading as _th

_probing = _th.Event()
_p0 = time.time()
def _probe_beat():
    while not _probing.wait(20):
        print(f'  [/director] {int(time.time()-_p0)}s elapsed — a grammar-constrained '
              'plan is thousands of tokens', flush=True)
_th.Thread(target=_probe_beat, daemon=True).start()
try:
    plan = requests.post(f'http://localhost:{API_PORT}/director', headers=AUTH,
                         json={'script': TEST_SCRIPT, 'style': 'finance'}, timeout=1800)
finally:
    _probing.set()
print('/director ', plan.status_code, f"{len(plan.json().get('scenes', []))} scenes" if plan.ok else plan.text[:200])
if not (chat.ok and gen.ok and plan.ok):
    raise RuntimeError('An endpoint failed — see the statuses above.')

PUBLIC_URL = None
if NGROK_AUTHTOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    PUBLIC_URL = ngrok.connect(API_PORT).public_url
    print(f'\n  Public URL : {PUBLIC_URL}')
    print(f'  API key    : {DIRECTOR_API_KEY}')
    print('\n  Put these in AETHER\'s .env, then restart the node server:')
    print(f'    QWEN_API_URL={PUBLIC_URL}')
    print(f'    QWEN_API_KEY={DIRECTOR_API_KEY}')
else:
    print('\nNGROK_AUTHTOKEN not set — reachable on localhost only.')
print('\nStage 10 PASSED')

## Stage 11 — Multi-domain benchmark

Optional, and slow: each plan is a few thousand grammar-constrained tokens, and llama.cpp answers one request at a time. Raise `N_DOMAINS` once Stage 7b has told you the rate.

In [ ]:
import time

N_DOMAINS = 4   # of 12; each plan takes roughly 1-4 min on 2x T4

DOMAINS = [
    ('finance',      'Inflation quietly reduces what your paycheck can buy. Central banks raise rates to cool prices.'),
    ('history',      'In 1944, Allied forces launched the largest amphibious invasion in history at Normandy.'),
    ('science',      'DNA carries the instructions for life. Every cell holds the same three billion base pairs.'),
    ('technology',   'Language models are trained on billions of tokens and predict the next word from learned patterns.'),
    ('medicine',     'Sleeping fewer than six hours a night significantly raises the risk of heart disease.'),
    ('cooking',      'The secret to risotto is patience: add warm stock one ladle at a time, stirring constantly.'),
    ('education',    'The Socratic method asks students to question assumptions rather than absorb answers.'),
    ('fitness',      'Interval training burns more calories in twenty minutes than an hour of steady jogging.'),
    ('business',     'Startups focused on customer problems are far likelier to find product-market fit.'),
    ('psychology',   'Cognitive dissonance is the discomfort of holding two conflicting beliefs at once.'),
    ('geography',    'The Amazon produces a fifth of the world oxygen and hosts a tenth of all known species.'),
    ('storytelling', 'Every story is a character who wants something, obstacles, and what the struggle reveals.'),
][:N_DOMAINS]

print(f'{"domain":<14}{"status":<8}{"secs":>7}{"scenes":>8}{"types":>7}')
print('-' * 46)

rows = []
for domain, script in DOMAINS:
    try:
        plan, latency = engine.generate_plan(script=script, style=domain)
        VideoPlan(**plan)
        scenes = plan['scenes']
        kinds = len({s['visualType'] for s in scenes})
        print(f'{domain:<14}{"ok":<8}{latency:>7.1f}{len(scenes):>8}{kinds:>7}')
        rows.append(True)
    except Exception as exc:
        print(f'{domain:<14}{"FAIL":<8}  {str(exc)[:40]}')
        rows.append(False)

print(f'\n{sum(rows)}/{len(rows)} passed')